<a href="https://colab.research.google.com/github/worasak2530seed/fast-hybrid-pro/blob/main/Fast_Hybrid_Pro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ============================================================
# FAST HYBRID PRO v2
# CLEAN RESEARCH / BACKTEST NOTEBOOK
# ============================================================

!pip -q install yfinance pandas numpy matplotlib

import warnings
warnings.filterwarnings("ignore")

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from dataclasses import dataclass, asdict

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 100)

print("✅ Setup complete")

In [ ]:
# ============================================================
# 2) CONFIGURATION
# ============================================================

CAPITAL = 100_000
MAX_RISK_PCT = 0.02
BOARD_LOT = 100
MIN_SCORE = 3
MIN_BARS = 200
WARMUP_BARS = 200
MAX_HOLDING_DAYS = 20

# ============================================================
# PRODUCTION STRATEGY — frozen parameters
# ============================================================
PRODUCTION_MIN_SCORE = 4
PRODUCTION_WEEKLY_CONFIRM = False
PRODUCTION_ATR_MULT = 1.2
PRODUCTION_TARGET2_PCT = 0.15
PRODUCTION_MAX_HOLDING_DAYS = 20

# ============================================================
# STOCK UNIVERSE FILTER — mandatory pre-scan gate
# Source: SET listed-company classification + market data
# ============================================================
UNIVERSE_MARKETS = {"SET", "mai"}

# User requirement: market cap > 5,000 MB
UNIVERSE_MIN_MARKET_CAP_MBAHT = 5_000

# User requirement: 20-day average trading value.
# 10 MB/day is retained as the minimum liquidity gate.
UNIVERSE_MIN_AVG20_VALUE_MBAHT = 10

# Fundamental price gate used in the original SeedGrowthTrading universe.
UNIVERSE_MIN_PRICE_BAHT = 1
UNIVERSE_MAX_PRICE_BAHT = 80

# Research/backtest MUST use the same fundamental universe gate as production.
RESEARCH_USE_FUNDAMENTAL_UNIVERSE_GATE = True

# Exclude SET/mai sectors corresponding to:
# banks, finance/hire-purchase, insurance, property/REITs, hotels/tourism.
UNIVERSE_EXCLUDED_SECTORS = {
    "BANK",
    "FIN",
    "INSUR",
    "PROP",
    "PF&REIT",
    "TOURISM",
}

# Only ordinary common stocks.
UNIVERSE_SECURITY_TYPE = "CS"

# The old hand-picked production universe is deliberately removed.
# Production symbols are rebuilt from the filtered SET/mai universe.
PRODUCTION_SYMBOLS = []

WATCHLIST = []

print("Universe filters:")
print(f"  Markets: {sorted(UNIVERSE_MARKETS)}")
print(f"  Market cap > {UNIVERSE_MIN_MARKET_CAP_MBAHT:,} MB")
print(f"  Avg 20-day trading value >= {UNIVERSE_MIN_AVG20_VALUE_MBAHT:,} MB/day")
print(f"  Price: {UNIVERSE_MIN_PRICE_BAHT}–{UNIVERSE_MAX_PRICE_BAHT} THB")
print(f"  Excluded sectors: {sorted(UNIVERSE_EXCLUDED_SECTORS)}")

@dataclass
class StrategyConfig:
    macd_fast: int = 4
    macd_slow: int = 12
    macd_signal: int = 9
    stoch_k: int = 8
    stoch_smooth: int = 3
    stoch_d: int = 3
    ema_fast: int = 5
    ema_mid: int = 10
    ema_long: int = 75
    weekly_ema_fast: int = 5
    weekly_ema_slow: int = 10
    atr_length: int = 14
    atr_stop_mult: float = 1.5
    bb_length: int = 20
    bb_mult: float = 2.0
    kc_mult: float = 1.5
    squeeze_lookback: int = 10
    volume_ma: int = 5
    volume_ratio_min: float = 1.2
    target1_pct: float = 0.07
    target2_pct: float = 0.15
    min_score: int = 3

CANONICAL = StrategyConfig(macd_fast=4, macd_slow=12, macd_signal=9)
FAST_MACD = StrategyConfig(macd_fast=6, macd_slow=13, macd_signal=5)

print("CANONICAL:", asdict(CANONICAL))
print("FAST MACD:", asdict(FAST_MACD))


## FUNDAMENTAL UNIVERSE GATE — RESEARCH + PRODUCTION

The same mandatory fundamental universe gate is applied **before indicators, signals, and backtests**.

- SET / mai only
- Common stocks only
- Market cap > 5,000 MB
- Average 20-day trading value >= 10 MB/day
- Price 1–80 THB
- Exclude BANK, FIN, INSUR, PROP, PF&REIT, TOURISM

This is a universe rule, not a strategy optimization parameter. It must not be changed by archived research variants.

**Important research integrity rule:** the current Yahoo market-cap value is a practical universe gate for the live scanner, but it is not a historical point-in-time market-cap series. Therefore this gate is not claimed to be a fully point-in-time survivorship-free fundamental backtest until historical market-cap snapshots are available. The research output must label this limitation rather than silently treating today's market cap as historical information.


In [ ]:
# ============================================================
# 3) DATA ENGINE + SET/mai UNIVERSE FILTER
# ============================================================

import io
import requests

REQUIRED_COLS = ["Open", "High", "Low", "Close", "Volume"]

SET_LISTED_COMPANIES_URL = (
    "https://www.set.or.th/dat/eod/listedcompany/static/"
    "listedCompanies_en_US.xls"
)

def load_set_listed_companies():
    """Load current SET/mai ordinary-share classification from SET."""
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; Fast-Hybrid-Pro/1.0)",
        "Accept": "text/html,application/vnd.ms-excel,application/octet-stream;q=0.9,*/*;q=0.8",
    }
    try:
        response = requests.get(SET_LISTED_COMPANIES_URL, headers=headers, timeout=30)
        response.raise_for_status()
        payload = response.content

        # Legacy SET URL may now return a web page instead of a workbook.
        tables = []
        try:
            tables = pd.read_html(io.BytesIO(payload))
        except Exception:
            tables = []

        # Fall back to the current SET securities table page.
        if not tables:
            html_url = "https://www.set.or.th/en/market/information/securities-list/thaiesg"
            html_response = requests.get(html_url, headers=headers, timeout=30)
            html_response.raise_for_status()
            tables = pd.read_html(io.StringIO(html_response.text))

        normalized = []
        for t in tables:
            t = t.copy()
            if isinstance(t.columns, pd.MultiIndex):
                t.columns = [
                    " ".join(str(x).strip() for x in col if str(x).strip() and str(x) != "nan")
                    for col in t.columns
                ]
            else:
                t.columns = [str(x).strip() for x in t.columns]
            normalized.append(t)

        raw = next(
            (
                t for t in normalized
                if any(str(c).strip().lower() == "symbol" for c in t.columns)
                and any(str(c).strip().lower() == "market" for c in t.columns)
                and any(str(c).strip().lower() == "sector" for c in t.columns)
            ),
            None,
        )
        if raw is None:
            raise ValueError("SET classification table not found.")

        raw.columns = [str(c).strip() for c in raw.columns]
        if "Industry Group" in raw.columns:
            raw["Industry"] = raw["Industry Group"]
        elif "Industry" not in raw.columns:
            raw["Industry"] = ""

        if "Company Name" in raw.columns:
            raw["Company"] = raw["Company Name"].astype(str).str.strip()
        elif "Company" not in raw.columns:
            raw["Company"] = ""

        # The current SET securities page contains funds/REITs as well as
        # ordinary listed companies. Ordinary company rows use PCL naming.
        raw["Company"] = raw["Company"].astype(str)
        raw = raw[
            raw["Company"].str.contains(
                r"PUBLIC COMPANY LIMITED|PUBLIC CO\.,?LTD\.?|PCL\.?$",
                case=False, regex=True, na=False
            )
        ].copy()

        out = raw.copy()
        out["Symbol"] = out["Symbol"].astype(str).str.strip().str.upper()
        out["Market"] = out["Market"].astype(str).str.strip()
        out["Industry"] = out["Industry"].astype(str).str.strip().str.upper()
        out["Sector"] = out["Sector"].astype(str).str.strip().str.upper()
        out = out[out["Market"].isin(UNIVERSE_MARKETS)].copy()
        out = out[out["Symbol"].ne("")].copy()
        if out.empty:
            raise ValueError("SET classification returned zero SET/mai ordinary-company rows.")
        return out.drop_duplicates("Symbol").reset_index(drop=True)

    except Exception as e:
        raise RuntimeError(
            "Cannot load SET listed-company classification. "
            f"Source={SET_LISTED_COMPANIES_URL}; error={e}"
        )

def normalize_ohlcv(df):
    if df is None or df.empty:
        return pd.DataFrame()
    out = df.copy()
    if isinstance(out.columns, pd.MultiIndex):
        out.columns = out.columns.get_level_values(0)
    out = out.loc[:, ~out.columns.duplicated()]
    missing = [c for c in REQUIRED_COLS if c not in out.columns]
    if missing:
        raise ValueError(f"Missing OHLCV columns: {missing}")
    out = out[REQUIRED_COLS].copy()
    out.index = pd.to_datetime(out.index)
    if getattr(out.index, "tz", None) is not None:
        out.index = out.index.tz_localize(None)
    for col in REQUIRED_COLS:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    out = out.dropna(subset=["Open", "High", "Low", "Close"])
    out = out[~out.index.duplicated(keep="last")].sort_index()
    return out

def download_stock(symbol, period="5y"):
    raw = yf.download(
        symbol,
        period=period,
        interval="1d",
        auto_adjust=False,
        progress=False,
        threads=False,
    )
    return normalize_ohlcv(raw)

def get_market_cap_baht(symbol):
    """Best-effort current market cap from Yahoo; UNKNOWN is rejected."""
    try:
        info = yf.Ticker(symbol).fast_info
        value = info.get("market_cap")
        if value is not None and np.isfinite(float(value)):
            return float(value)
    except Exception:
        pass
    try:
        value = yf.Ticker(symbol).info.get("marketCap")
        if value is not None and np.isfinite(float(value)):
            return float(value)
    except Exception:
        pass
    return np.nan

def build_filtered_universe():
    listed = load_set_listed_companies()

    # Hard sector exclusion uses SET's official sector codes.
    listed = listed[~listed["Sector"].isin(UNIVERSE_EXCLUDED_SECTORS)].copy()

    symbols = listed["Symbol"].tolist()
    yf_symbols = [f"{symbol}.BK" for symbol in symbols]

    print(f"SET/mai candidates after sector exclusion: {len(listed)}")
    print("Downloading one batched 3-month OHLCV snapshot for the universe...")

    raw = yf.download(
        yf_symbols,
        period="3mo",
        interval="1d",
        auto_adjust=False,
        progress=False,
        threads=True,
        group_by="ticker",
    )

    rows = []
    for symbol in symbols:
        yf_symbol = f"{symbol}.BK"
        try:
            if isinstance(raw.columns, pd.MultiIndex):
                if yf_symbol not in raw.columns.get_level_values(0):
                    continue
                df = raw[yf_symbol].copy()
            else:
                df = raw.copy()

            df = normalize_ohlcv(df)
            if df.empty or len(df) < 20:
                continue

            recent = df.tail(20).copy()
            avg20_value_baht = float((recent["Close"] * recent["Volume"]).mean())
            last_price_baht = float(recent["Close"].iloc[-1])

            # Fundamental universe gates that can be evaluated from the market snapshot
            # are applied before the slower market-cap lookup.
            if not (
                np.isfinite(avg20_value_baht)
                and avg20_value_baht / 1_000_000 >= UNIVERSE_MIN_AVG20_VALUE_MBAHT
            ):
                continue
            if not (
                np.isfinite(last_price_baht)
                and UNIVERSE_MIN_PRICE_BAHT <= last_price_baht <= UNIVERSE_MAX_PRICE_BAHT
            ):
                continue

            market_cap_baht = get_market_cap_baht(yf_symbol)
            if not np.isfinite(market_cap_baht):
                continue

            rows.append({
                "Symbol": yf_symbol,
                "Market": row["Market"] if False else listed.loc[listed["Symbol"].eq(symbol), "Market"].iloc[0],
                "Industry": listed.loc[listed["Symbol"].eq(symbol), "Industry"].iloc[0],
                "Sector": listed.loc[listed["Symbol"].eq(symbol), "Sector"].iloc[0],
                "MarketCapMB": market_cap_baht / 1_000_000,
                "Avg20TradingValueMB": avg20_value_baht / 1_000_000,
                "LastPrice": last_price_baht,
            })
        except Exception as e:
            print(f"Universe data skipped {yf_symbol}: {e}")

    audit = pd.DataFrame(rows)
    if audit.empty:
        raise RuntimeError("Filtered universe is empty; refusing to run scanner.")

    audit["PassMarketCap"] = audit["MarketCapMB"] > UNIVERSE_MIN_MARKET_CAP_MBAHT
    audit["PassAvg20Value"] = audit["Avg20TradingValueMB"] >= UNIVERSE_MIN_AVG20_VALUE_MBAHT
    audit["PassPrice"] = audit["LastPrice"].between(
        UNIVERSE_MIN_PRICE_BAHT, UNIVERSE_MAX_PRICE_BAHT, inclusive="both"
    )
    audit["PassUniverse"] = (
        audit["PassMarketCap"]
        & audit["PassAvg20Value"]
        & audit["PassPrice"]
    )

    filtered = audit[audit["PassUniverse"]].copy()
    filtered = filtered.sort_values(
        ["Avg20TradingValueMB", "MarketCapMB"],
        ascending=[False, False],
    ).reset_index(drop=True)

    print("\n====================================")
    print("FILTERED SET/mai FUNDAMENTAL UNIVERSE")
    print("====================================")
    print(f"Rows with market-cap data : {len(audit)}")
    print(f"Market cap pass            : {int(audit['PassMarketCap'].sum())}")
    print(f"20D value pass             : {int(audit['PassAvg20Value'].sum())}")
    print(f"Price 1–80 pass            : {int(audit['PassPrice'].sum())}")
    print(f"Final universe             : {len(filtered)}")
    display(filtered.head(100))

    return filtered, audit

filtered_universe, universe_audit = build_filtered_universe()

PRODUCTION_SYMBOLS = filtered_universe["Symbol"].tolist()
WATCHLIST = PRODUCTION_SYMBOLS.copy()

print(f"\nProduction symbols loaded: {len(PRODUCTION_SYMBOLS)}")
print(PRODUCTION_SYMBOLS)

data = {}
errors = {}

print("\n🚀 Downloading filtered-universe market data...")

for symbol in WATCHLIST:
    try:
        df = download_stock(symbol)
        if len(df) < MIN_BARS:
            errors[symbol] = f"Only {len(df)} bars (minimum {MIN_BARS})"
        else:
            data[symbol] = df
            print(f"✅ {symbol:<12} {len(df):>5} bars")
    except Exception as e:
        errors[symbol] = str(e)
        print(f"❌ {symbol}: {e}")

print("\n====================================")
print(f"Loaded: {len(data)} / {len(WATCHLIST)}")
print("====================================")

if errors:
    print("\n⚠️ Errors / insufficient data:")
    for symbol, error in errors.items():
        print(f"{symbol}: {error}")


In [ ]:

# ============================================================
# 4) DAILY INDICATORS
# ============================================================

def wilder_rma(series, length):
    """
    Wilder RMA
    ใช้แนวคิดเดียวกับ TradingView ta.rma()
    """

    return series.ewm(
        alpha=1 / length,
        adjust=False,
        min_periods=length
    ).mean()


def calculate_atr(df, length=14):

    prev_close = df["Close"].shift(1)

    tr1 = df["High"] - df["Low"]
    tr2 = (df["High"] - prev_close).abs()
    tr3 = (df["Low"] - prev_close).abs()

    true_range = pd.concat(
        [tr1, tr2, tr3],
        axis=1
    ).max(axis=1)

    atr = wilder_rma(
        true_range,
        length
    )

    return true_range, atr


def calculate_daily_indicators(df):

    out = df.copy()

    close = out["Close"]
    high = out["High"]
    low = out["Low"]
    volume = out["Volume"]

    # --------------------------------------------------------
    # EMA 5 / 10 / 75
    # --------------------------------------------------------

    out["EMA5"] = close.ewm(
        span=5,
        adjust=False
    ).mean()

    out["EMA10"] = close.ewm(
        span=10,
        adjust=False
    ).mean()

    out["EMA75"] = close.ewm(
        span=75,
        adjust=False
    ).mean()

    # --------------------------------------------------------
    # MACD 4 / 12 / 9
    # --------------------------------------------------------

    ema_fast = close.ewm(
        span=4,
        adjust=False
    ).mean()

    ema_slow = close.ewm(
        span=12,
        adjust=False
    ).mean()

    out["MACD"] = ema_fast - ema_slow

    out["MACD_SIGNAL"] = out["MACD"].ewm(
        span=9,
        adjust=False
    ).mean()

    out["MACD_HIST"] = (
        out["MACD"] - out["MACD_SIGNAL"]
    )

    # --------------------------------------------------------
    # STOCHASTIC 8 / 3 / 3
    # --------------------------------------------------------

    lowest_8 = low.rolling(
        window=8,
        min_periods=8
    ).min()

    highest_8 = high.rolling(
        window=8,
        min_periods=8
    ).max()

    price_range = highest_8 - lowest_8

    stoch_raw = (
        100
        * (close - lowest_8)
        / price_range.replace(0, np.nan)
    )

    out["STOCH_K"] = stoch_raw.rolling(
        window=3,
        min_periods=3
    ).mean()

    out["STOCH_D"] = out["STOCH_K"].rolling(
        window=3,
        min_periods=3
    ).mean()

    # --------------------------------------------------------
    # ATR 14 — WILDER
    # --------------------------------------------------------

    out["TR"], out["ATR14"] = calculate_atr(
        out,
        length=14
    )

    # --------------------------------------------------------
    # BOLLINGER BAND 20 / 2
    # --------------------------------------------------------

    bb_mid = close.rolling(
        window=20,
        min_periods=20
    ).mean()

    bb_std = close.rolling(
        window=20,
        min_periods=20
    ).std()

    out["BB_MID"] = bb_mid

    out["BB_UPPER"] = (
        bb_mid + (2.0 * bb_std)
    )

    out["BB_LOWER"] = (
        bb_mid - (2.0 * bb_std)
    )

    # --------------------------------------------------------
    # KELTNER CHANNEL
    # --------------------------------------------------------

    out["KC_MID"] = bb_mid

    out["KC_UPPER"] = (
        bb_mid + (1.5 * out["ATR14"])
    )

    out["KC_LOWER"] = (
        bb_mid - (1.5 * out["ATR14"])
    )

    # --------------------------------------------------------
    # SQUEEZE
    # --------------------------------------------------------

    out["SQUEEZE"] = (
        (out["BB_UPPER"] <= out["KC_UPPER"])
        &
        (out["BB_LOWER"] >= out["KC_LOWER"])
    )

    # --------------------------------------------------------
    # SQUEEZE RELEASE
    # --------------------------------------------------------

    out["SQUEEZE_RELEASE"] = (
        (~out["SQUEEZE"])
        &
        (out["SQUEEZE"].shift(1).fillna(False))
    )

    # --------------------------------------------------------
    # BARS SINCE LAST SQUEEZE
    # --------------------------------------------------------

    bars_since_squeeze = []

    last_squeeze_index = None

    for i, is_squeeze in enumerate(
        out["SQUEEZE"].fillna(False)
    ):

        if is_squeeze:

            last_squeeze_index = i
            bars_since_squeeze.append(0)

        elif last_squeeze_index is None:

            bars_since_squeeze.append(np.nan)

        else:

            bars_since_squeeze.append(
                i - last_squeeze_index
            )

    out["BARS_SINCE_SQUEEZE"] = (
        bars_since_squeeze
    )

    out["SQUEEZE_RECENT"] = (
        out["BARS_SINCE_SQUEEZE"] <= 10
    )

    # --------------------------------------------------------
    # VOLUME
    # --------------------------------------------------------

    out["VOLUME_MA5"] = volume.rolling(
        window=5,
        min_periods=5
    ).mean()

    out["VOLUME_RATIO"] = (
        volume
        / out["VOLUME_MA5"].replace(0, np.nan)
    )

    # --------------------------------------------------------
    # CANDLE BODY
    # --------------------------------------------------------

    out["BODY"] = (
        close - out["Open"]
    ).abs()

    out["BODY_ATR_RATIO"] = (
        out["BODY"]
        / out["ATR14"].replace(0, np.nan)
    )

    return out


# ============================================================
# APPLY TO ALL SYMBOLS
# ============================================================

daily_data = {}
indicator_errors = {}

print("📊 Calculating daily indicators...\n")

for symbol, df in data.items():

    try:

        result = calculate_daily_indicators(df)

        daily_data[symbol] = result

        print(
            f"✅ {symbol:<12} "
            f"{len(result):>5} bars"
        )

    except Exception as e:

        indicator_errors[symbol] = str(e)

        print(
            f"❌ {symbol}: {e}"
        )


print("\n====================================")
print(
    f"Daily indicator data: "
    f"{len(daily_data)} / {len(data)}"
)
print("====================================")


if indicator_errors:

    print("\n⚠️ Indicator errors:")

    for symbol, error in indicator_errors.items():

        print(
            f"{symbol}: {error}"
        )


# ============================================================
# QUICK VALIDATION
# ============================================================

if daily_data:

    sample_symbol = next(iter(daily_data))

    sample = daily_data[sample_symbol]

    print(
        f"\n🔎 Quick validation: {sample_symbol}"
    )

    display(
        sample[
            [
                "Close",
                "EMA5",
                "EMA10",
                "EMA75",
                "MACD",
                "MACD_SIGNAL",
                "MACD_HIST",
                "STOCH_K",
                "STOCH_D",
                "ATR14",
                "SQUEEZE",
                "SQUEEZE_RELEASE",
                "VOLUME_RATIO"
            ]
        ].tail(5)
    )

In [ ]:

# ============================================================
# 5) WEEKLY INDICATORS
# ============================================================

def calculate_weekly_indicators(df):

    daily = df.copy()

    # --------------------------------------------------------
    # Daily -> Weekly
    #
    # Week ends on Friday.
    # --------------------------------------------------------

    weekly = daily.resample("W-FRI").agg({
        "Open": "first",
        "High": "max",
        "Low": "min",
        "Close": "last",
        "Volume": "sum"
    })

    weekly = weekly.dropna(
        subset=[
            "Open",
            "High",
            "Low",
            "Close"
        ]
    )

    close = weekly["Close"]

    # --------------------------------------------------------
    # WEEKLY EMA 5 / 10
    # --------------------------------------------------------

    weekly["W_EMA5"] = close.ewm(
        span=5,
        adjust=False
    ).mean()

    weekly["W_EMA10"] = close.ewm(
        span=10,
        adjust=False
    ).mean()

    # --------------------------------------------------------
    # WEEKLY MACD 4 / 12 / 9
    # --------------------------------------------------------

    weekly_ema_fast = close.ewm(
        span=4,
        adjust=False
    ).mean()

    weekly_ema_slow = close.ewm(
        span=12,
        adjust=False
    ).mean()

    weekly["W_MACD"] = (
        weekly_ema_fast
        - weekly_ema_slow
    )

    weekly["W_MACD_SIGNAL"] = (
        weekly["W_MACD"].ewm(
            span=9,
            adjust=False
        ).mean()
    )

    weekly["W_MACD_HIST"] = (
        weekly["W_MACD"]
        - weekly["W_MACD_SIGNAL"]
    )

    # --------------------------------------------------------
    # IMPORTANT:
    #
    # A weekly candle ending Friday becomes available
    # only from the next trading day.
    #
    # Example:
    #
    # Week ending 2026-09-25
    # -> usable from 2026-09-28
    #
    # This prevents Friday from seeing its own completed
    # weekly candle during the Friday signal.
    # --------------------------------------------------------

    weekly_features = weekly[
        [
            "W_EMA5",
            "W_EMA10",
            "W_MACD",
            "W_MACD_SIGNAL",
            "W_MACD_HIST"
        ]
    ].copy()

    # Move availability date to next Monday.
    #
    # W-FRI + 3 days = Monday
    #
    # If Monday is a market holiday, forward-fill will keep
    # the last completed weekly value until the next trading day.
    weekly_features.index = (
        weekly_features.index
        + pd.Timedelta(days=3)
    )

    # --------------------------------------------------------
    # Map completed weekly values to daily data
    # --------------------------------------------------------

    daily_index = daily.index

    mapped_weekly = weekly_features.reindex(
        daily_index,
        method="ffill"
    )

    # --------------------------------------------------------
    # Combine
    # --------------------------------------------------------

    out = daily.copy()

    for col in mapped_weekly.columns:
        out[col] = mapped_weekly[col]

    return out, weekly


# ============================================================
# APPLY TO ALL SYMBOLS
# ============================================================

weekly_data = {}
combined_data = {}
weekly_errors = {}

print("📅 Calculating weekly indicators...\n")

for symbol, df in daily_data.items():

    try:

        combined, weekly = calculate_weekly_indicators(df)

        combined_data[symbol] = combined
        weekly_data[symbol] = weekly

        print(
            f"✅ {symbol:<12} "
            f"Daily: {len(combined):>5} | "
            f"Weekly: {len(weekly):>4}"
        )

    except Exception as e:

        weekly_errors[symbol] = str(e)

        print(
            f"❌ {symbol}: {e}"
        )


print("\n====================================")
print(
    f"Weekly indicator data: "
    f"{len(combined_data)} / {len(daily_data)}"
)
print("====================================")


if weekly_errors:

    print("\n⚠️ Weekly indicator errors:")

    for symbol, error in weekly_errors.items():

        print(
            f"{symbol}: {error}"
        )


# ============================================================
# VALIDATION
# ============================================================

if combined_data:

    sample_symbol = next(iter(combined_data))

    sample = combined_data[sample_symbol]

    print(
        f"\n🔎 Weekly validation: {sample_symbol}"
    )

    display(
        sample[
            [
                "Close",
                "W_EMA5",
                "W_EMA10",
                "W_MACD",
                "W_MACD_SIGNAL",
                "W_MACD_HIST"
            ]
        ].tail(15)
    )

In [ ]:

# ============================================================
# 6) BUILD FEATURE DATASETS
# ============================================================

feature_data = {}

feature_errors = {}


for symbol, df in combined_data.items():

    try:

        out = df.copy()

        # ----------------------------------------------------
        # Basic validity
        # ----------------------------------------------------

        required = [
            "Open",
            "High",
            "Low",
            "Close",
            "Volume",

            "EMA5",
            "EMA10",
            "EMA75",

            "MACD",
            "MACD_SIGNAL",
            "MACD_HIST",

            "STOCH_K",
            "STOCH_D",

            "ATR14",

            "BB_UPPER",
            "BB_LOWER",

            "KC_UPPER",
            "KC_LOWER",

            "SQUEEZE",
            "SQUEEZE_RELEASE",
            "SQUEEZE_RECENT",

            "VOLUME_RATIO",

            "W_EMA5",
            "W_EMA10",

            "W_MACD",
            "W_MACD_SIGNAL",
            "W_MACD_HIST"
        ]

        missing = [
            col
            for col in required
            if col not in out.columns
        ]

        if missing:

            raise ValueError(
                f"Missing columns: {missing}"
            )

        # ----------------------------------------------------
        # Remove rows that cannot produce indicators
        #
        # IMPORTANT:
        # We do NOT drop rows globally before calculating
        # indicators. All indicators have already been
        # calculated using the full historical dataset.
        # ----------------------------------------------------

        out = out.dropna(
            subset=[
                "EMA75",
                "ATR14",
                "MACD",
                "MACD_SIGNAL",
                "MACD_HIST",
                "STOCH_K",
                "STOCH_D",
                "W_EMA5",
                "W_EMA10",
                "W_MACD",
                "W_MACD_SIGNAL",
                "W_MACD_HIST",
                "VOLUME_RATIO"
            ]
        )

        # ----------------------------------------------------
        # Ensure chronological order
        # ----------------------------------------------------

        out = out.sort_index()

        # ----------------------------------------------------
        # Ensure unique dates
        # ----------------------------------------------------

        out = out[
            ~out.index.duplicated(
                keep="last"
            )
        ]

        # ----------------------------------------------------
        # Create previous-bar values
        #
        # Used later for signal confirmation.
        # ----------------------------------------------------

        out["PREV_CLOSE"] = (
            out["Close"].shift(1)
        )

        out["PREV_MACD"] = (
            out["MACD"].shift(1)
        )

        out["PREV_MACD_SIGNAL"] = (
            out["MACD_SIGNAL"].shift(1)
        )

        out["PREV_MACD_HIST"] = (
            out["MACD_HIST"].shift(1)
        )

        out["PREV_STOCH_K"] = (
            out["STOCH_K"].shift(1)
        )

        out["PREV_STOCH_D"] = (
            out["STOCH_D"].shift(1)
        )

        # ----------------------------------------------------
        # Daily trend state
        # ----------------------------------------------------

        out["TREND_BULL"] = (
            (out["Close"] > out["EMA5"])
            &
            (out["EMA5"] > out["EMA10"])
            &
            (out["EMA10"] > out["EMA75"])
        )

        # ----------------------------------------------------
        # Weekly trend state
        # ----------------------------------------------------

        out["WEEKLY_TREND_BULL"] = (
            out["W_EMA5"] > out["W_EMA10"]
        )

        # ----------------------------------------------------
        # Daily MACD bullish state
        # ----------------------------------------------------

        out["MACD_BULL"] = (
            out["MACD"] > out["MACD_SIGNAL"]
        )

        out["MACD_HIST_RISING"] = (
            out["MACD_HIST"]
            > out["PREV_MACD_HIST"]
        )

        # ----------------------------------------------------
        # Weekly MACD bullish state
        # ----------------------------------------------------

        out["WEEKLY_MACD_BULL"] = (
            out["W_MACD"] > out["W_MACD_SIGNAL"]
        )

        out["WEEKLY_MACD_HIST_RISING"] = (
            out["W_MACD_HIST"]
            > out["W_MACD_HIST"].shift(1)
        )

        # ----------------------------------------------------
        # Stochastic bullish state
        # ----------------------------------------------------

        out["STOCH_BULL"] = (
            (out["STOCH_K"] > out["STOCH_D"])
            &
            (out["STOCH_K"] < 80)
        )

        # ----------------------------------------------------
        # Volume confirmation
        # ----------------------------------------------------

        out["VOLUME_CONFIRM"] = (
            out["VOLUME_RATIO"]
            >= 1.2
        )

        # ----------------------------------------------------
        # Early-entry / squeeze state
        # ----------------------------------------------------

        out["EARLY_ENTRY_ZONE"] = (
            out["SQUEEZE"]
            |
            out["SQUEEZE_RELEASE"]
            |
            out["SQUEEZE_RECENT"]
        )

        # ----------------------------------------------------
        # Target prices
        # ----------------------------------------------------

        out["TARGET1"] = (
            out["Close"] * 1.07
        )

        out["TARGET2"] = (
            out["Close"] * 1.15
        )

        # ----------------------------------------------------
        # Final dataset
        # ----------------------------------------------------

        feature_data[symbol] = out

    except Exception as e:

        feature_errors[symbol] = str(e)


print("📦 Building feature datasets...\n")

for symbol, df in feature_data.items():

    print(
        f"✅ {symbol:<12} "
        f"{len(df):>5} usable bars"
    )


print("\n====================================")
print(
    f"Feature datasets: "
    f"{len(feature_data)} / {len(combined_data)}"
)
print("====================================")


if feature_errors:

    print("\n⚠️ Feature errors:")

    for symbol, error in feature_errors.items():

        print(
            f"{symbol}: {error}"
        )


# ============================================================
# VALIDATION
# ============================================================

if feature_data:

    sample_symbol = next(iter(feature_data))

    sample = feature_data[sample_symbol]

    print(
        f"\n🔎 Feature validation: {sample_symbol}"
    )

    display(
        sample[
            [
                "Close",
                "EMA5",
                "EMA10",
                "EMA75",
                "MACD",
                "MACD_SIGNAL",
                "MACD_HIST",
                "W_EMA5",
                "W_EMA10",
                "W_MACD",
                "W_MACD_SIGNAL",
                "W_MACD_HIST",
                "TREND_BULL",
                "WEEKLY_TREND_BULL",
                "MACD_BULL",
                "STOCH_BULL",
                "VOLUME_CONFIRM",
                "EARLY_ENTRY_ZONE"
            ]
        ].tail(10)
    )

In [ ]:

# ============================================================
# 7) LEGACY SCORE
# ============================================================
#
# วัตถุประสงค์:
# - เก็บสูตร FAST HYBRID PRO เดิมไว้เป็น Baseline
# - ใช้เปรียบเทียบกับ Canonical Research Score ใน Cell 8
# - ยังไม่ทำ Backtest และยังไม่เปิด Position
#
# LEGACY 5-POINT SCORE
# 1. MACD 6/13/5 + Histogram Rising
# 2. Stochastic 8/3/3 : K > D และ K < 80
# 3. Trend: Close > EMA20 > EMA50
# 4. Candle Body / ATR >= 0.8
# 5. Volatility: BB breakout OR squeeze release
#
# หมายเหตุ:
# - Cell 4 ใช้ EMA5/10/75 และ MACD 4/12/9 เป็น canonical data
# - ดังนั้น Legacy MACD จะคำนวณใหม่ตรงนี้ด้วย 6/13/5
# - EMA20/EMA50 ก็จะคำนวณใหม่เพื่อรักษาสูตร Legacy เดิม
# ============================================================


def calculate_legacy_score(df):
    out = df.copy()

    # --------------------------------------------------------
    # 1) LEGACY MACD 6/13/5
    # --------------------------------------------------------
    close = out["Close"]

    legacy_ema_fast = close.ewm(
        span=6,
        adjust=False
    ).mean()

    legacy_ema_slow = close.ewm(
        span=13,
        adjust=False
    ).mean()

    out["LEGACY_MACD"] = (
        legacy_ema_fast - legacy_ema_slow
    )

    out["LEGACY_MACD_SIGNAL"] = (
        out["LEGACY_MACD"]
        .ewm(span=5, adjust=False)
        .mean()
    )

    out["LEGACY_MACD_HIST"] = (
        out["LEGACY_MACD"]
        - out["LEGACY_MACD_SIGNAL"]
    )

    out["LEGACY_MACD_HIST_PREV"] = (
        out["LEGACY_MACD_HIST"].shift(1)
    )

    out["LEGACY_MACD_BULL"] = (
        (out["LEGACY_MACD"] > out["LEGACY_MACD_SIGNAL"])
        &
        (
            out["LEGACY_MACD_HIST"]
            > out["LEGACY_MACD_HIST_PREV"]
        )
    )


    # --------------------------------------------------------
    # 2) LEGACY STOCHASTIC 8/3/3
    # --------------------------------------------------------
    lowest_8 = out["Low"].rolling(
        window=8,
        min_periods=8
    ).min()

    highest_8 = out["High"].rolling(
        window=8,
        min_periods=8
    ).max()

    price_range = (
        highest_8 - lowest_8
    ).replace(0, np.nan)

    stoch_raw = (
        100
        * (close - lowest_8)
        / price_range
    )

    out["LEGACY_STOCH_K"] = (
        stoch_raw
        .rolling(window=3, min_periods=3)
        .mean()
    )

    out["LEGACY_STOCH_D"] = (
        out["LEGACY_STOCH_K"]
        .rolling(window=3, min_periods=3)
        .mean()
    )

    out["LEGACY_STOCH_BULL"] = (
        (out["LEGACY_STOCH_K"] > out["LEGACY_STOCH_D"])
        &
        (out["LEGACY_STOCH_K"] < 80)
    )


    # --------------------------------------------------------
    # 3) LEGACY TREND
    # Close > EMA20 > EMA50
    # --------------------------------------------------------
    out["LEGACY_EMA20"] = (
        close.ewm(
            span=20,
            adjust=False
        ).mean()
    )

    out["LEGACY_EMA50"] = (
        close.ewm(
            span=50,
            adjust=False
        ).mean()
    )

    out["LEGACY_TREND_BULL"] = (
        (close > out["LEGACY_EMA20"])
        &
        (out["LEGACY_EMA20"] > out["LEGACY_EMA50"])
    )


    # --------------------------------------------------------
    # 4) CANDLE BODY / ATR >= 0.8
    # --------------------------------------------------------
    #
    # ใช้ ATR14 จาก Cell 4
    # Body = abs(Close - Open)
    #
    # เงื่อนไขเดิม:
    # Body / ATR >= 0.8
    # --------------------------------------------------------

    out["LEGACY_BODY_ATR_RATIO"] = (
        (out["Close"] - out["Open"]).abs()
        / out["ATR14"].replace(0, np.nan)
    )

    out["LEGACY_CANDLE_BULL"] = (
        out["LEGACY_BODY_ATR_RATIO"] >= 0.8
    )


    # --------------------------------------------------------
    # 5) VOLATILITY
    # BB Breakout OR Squeeze Release
    # --------------------------------------------------------
    #
    # BB breakout:
    # Close > Upper BB
    #
    # Squeeze release:
    # SQUEEZE วันนี้ False
    # และเมื่อวานเป็น True
    # --------------------------------------------------------

    out["LEGACY_BB_BREAKOUT"] = (
        out["Close"] > out["BB_UPPER"]
    )

    out["LEGACY_SQUEEZE_RELEASE"] = (
        (~out["SQUEEZE"])
        &
        (out["SQUEEZE"].shift(1).fillna(False))
    )

    out["LEGACY_VOLATILITY_BULL"] = (
        out["LEGACY_BB_BREAKOUT"]
        |
        out["LEGACY_SQUEEZE_RELEASE"]
    )


    # --------------------------------------------------------
    # LEGACY SCORE
    # --------------------------------------------------------

    score_components = [
        "LEGACY_MACD_BULL",
        "LEGACY_STOCH_BULL",
        "LEGACY_TREND_BULL",
        "LEGACY_CANDLE_BULL",
        "LEGACY_VOLATILITY_BULL",
    ]

    out["LEGACY_SCORE"] = (
        out[score_components]
        .fillna(False)
        .astype(int)
        .sum(axis=1)
    )


    # --------------------------------------------------------
    # Signal classification
    # --------------------------------------------------------
    #
    # Legacy เดิมใช้ 5 คะแนนเต็ม
    #
    # >= 3 = ผ่าน baseline
    # < 3  = ไม่ผ่าน
    # --------------------------------------------------------

    out["LEGACY_SIGNAL"] = (
        out["LEGACY_SCORE"] >= 3
    )


    return out


# ============================================================
# APPLY TO ALL SYMBOLS
# ============================================================

legacy_data = {}
legacy_errors = {}

print("🧮 Calculating Legacy Score...\n")

for symbol, df in feature_data.items():
    try:
        result = calculate_legacy_score(df)

        legacy_data[symbol] = result

        print(
            f"✅ {symbol:<12} "
            f"{len(result):>5} bars"
        )

    except Exception as e:
        legacy_errors[symbol] = str(e)

        print(
            f"❌ {symbol}: {e}"
        )


# ============================================================
# SUMMARY
# ============================================================

print("\n====================================")
print(
    f"Legacy datasets: "
    f"{len(legacy_data)} / {len(feature_data)}"
)
print("====================================")


if legacy_errors:
    print("\n⚠️ Legacy calculation errors:")

    for symbol, error in legacy_errors.items():
        print(f"{symbol}: {error}")


# ============================================================
# VALIDATION
# ============================================================

if legacy_data:

    sample_symbol = next(iter(legacy_data))
    sample = legacy_data[sample_symbol]

    print(
        f"\n🔎 Legacy validation: "
        f"{sample_symbol}"
    )

    display(
        sample[
            [
                "Close",
                "LEGACY_EMA20",
                "LEGACY_EMA50",

                "LEGACY_MACD",
                "LEGACY_MACD_SIGNAL",
                "LEGACY_MACD_HIST",

                "LEGACY_STOCH_K",
                "LEGACY_STOCH_D",

                "LEGACY_BODY_ATR_RATIO",

                "LEGACY_BB_BREAKOUT",
                "LEGACY_SQUEEZE_RELEASE",

                "LEGACY_MACD_BULL",
                "LEGACY_STOCH_BULL",
                "LEGACY_TREND_BULL",
                "LEGACY_CANDLE_BULL",
                "LEGACY_VOLATILITY_BULL",

                "LEGACY_SCORE",
                "LEGACY_SIGNAL",
            ]
        ].tail(10)
    )


# ============================================================
# SCORE DISTRIBUTION
# ============================================================

if legacy_data:

    all_scores = pd.concat(
        [
            df["LEGACY_SCORE"]
            for df in legacy_data.values()
        ]
    )

    print("\n📊 Legacy Score Distribution")

    display(
        all_scores
        .value_counts()
        .sort_index()
        .rename("bars")
        .to_frame()
    )

    print(
        f"\nBars with Legacy Score >= 3: "
        f"{int((all_scores >= 3).sum()):,}"
    )

    print(
        f"Total usable bars: "
        f"{len(all_scores):,}"
    )

    print(
        f"Signal rate: "
        f"{(all_scores >= 3).mean() * 100:.2f}%"
    )

In [ ]:

# ============================================================
# 8) CANONICAL RESEARCH SCORE
# ============================================================
#
# PURPOSE
# ------------------------------------------------------------
# สร้างคะแนนระบบใหม่เพื่อใช้เปรียบเทียบกับ Legacy Score
#
# CANONICAL INDICATORS
#   DAY MACD      = 4 / 12 / 9
#   STOCHASTIC    = 8 / 3 / 3
#   DAY EMA       = 5 / 10 / 75
#   WEEK EMA      = 5 / 10
#   WEEK MACD     = 4 / 12 / 9
#
# RESEARCH SCORE = 5 POINTS
#
# 1) MACD
#    DAY MACD > Signal
#    AND Histogram > 0
#
# 2) STOCHASTIC
#    K > D
#    AND K < 80
#
# 3) DAY TREND
#    Close > EMA5 > EMA10 > EMA75
#
# 4) EARLY ENTRY
#    Squeeze OR Squeeze Release
#
# 5) VOLUME
#    Volume Ratio >= 1.2
#
# PASS:
#    Score >= 3
#
# IMPORTANT
# ------------------------------------------------------------
# Cell นี้เป็น Research Baseline
# ไม่ใช่ Production Weighted Score
# ============================================================


def calculate_canonical_research_score(df):

    out = df.copy()

    # --------------------------------------------------------
    # 1) DAY MACD 4/12/9
    # --------------------------------------------------------

    out["CANONICAL_MACD_BULL"] = (
        (out["MACD"] > out["MACD_SIGNAL"])
        &
        (out["MACD_HIST"] > 0)
    )


    # --------------------------------------------------------
    # 2) STOCHASTIC 8/3/3
    # --------------------------------------------------------

    out["CANONICAL_STOCH_BULL"] = (
        (out["STOCH_K"] > out["STOCH_D"])
        &
        (out["STOCH_K"] < 80)
    )


    # --------------------------------------------------------
    # 3) DAY TREND
    #
    # Close > EMA5 > EMA10 > EMA75
    # --------------------------------------------------------

    out["CANONICAL_TREND_BULL"] = (
        (out["Close"] > out["EMA5"])
        &
        (out["EMA5"] > out["EMA10"])
        &
        (out["EMA10"] > out["EMA75"])
    )


    # --------------------------------------------------------
    # 4) EARLY ENTRY
    #
    # ใช้ Squeeze หรือ Squeeze Release
    #
    # ไม่ใช้ SQUEEZE_RECENT ในคะแนนหลัก
    # เพื่อไม่ให้ signal window กว้างเกินไป
    # --------------------------------------------------------

    out["CANONICAL_EARLY_ENTRY"] = (
        out["SQUEEZE"]
        |
        out["SQUEEZE_RELEASE"]
    )


    # --------------------------------------------------------
    # 5) VOLUME CONFIRMATION
    # --------------------------------------------------------

    out["CANONICAL_VOLUME_CONFIRM"] = (
        out["VOLUME_RATIO"] >= 1.2
    )


    # --------------------------------------------------------
    # SCORE COMPONENTS
    # --------------------------------------------------------

    canonical_components = [
        "CANONICAL_MACD_BULL",
        "CANONICAL_STOCH_BULL",
        "CANONICAL_TREND_BULL",
        "CANONICAL_EARLY_ENTRY",
        "CANONICAL_VOLUME_CONFIRM",
    ]

    out["CANONICAL_SCORE"] = (
        out[canonical_components]
        .fillna(False)
        .astype(int)
        .sum(axis=1)
    )


    # --------------------------------------------------------
    # SIGNAL
    # --------------------------------------------------------

    out["CANONICAL_SIGNAL"] = (
        out["CANONICAL_SCORE"] >= 3
    )


    # --------------------------------------------------------
    # ADDITIONAL INFORMATION
    # --------------------------------------------------------
    #
    # Weekly confirmation ไม่ถูกนับเป็นคะแนนเพิ่ม
    # ใน Research Score
    #
    # เก็บไว้เพื่อใช้วิเคราะห์ภายหลัง
    # --------------------------------------------------------

    out["CANONICAL_WEEKLY_TREND"] = (
        out["W_EMA5"] > out["W_EMA10"]
    )

    out["CANONICAL_WEEKLY_MACD"] = (
        out["W_MACD"] > out["W_MACD_SIGNAL"]
    )

    out["CANONICAL_WEEKLY_CONFIRM"] = (
        out["CANONICAL_WEEKLY_TREND"]
        &
        out["CANONICAL_WEEKLY_MACD"]
    )


    return out


# ============================================================
# APPLY TO ALL SYMBOLS
# ============================================================

canonical_data = {}
canonical_errors = {}

print("🧮 Calculating Canonical Research Score...\n")

for symbol, df in feature_data.items():

    try:

        result = calculate_canonical_research_score(df)

        canonical_data[symbol] = result

        print(
            f"✅ {symbol:<12} "
            f"{len(result):>5} bars"
        )

    except Exception as e:

        canonical_errors[symbol] = str(e)

        print(
            f"❌ {symbol}: {e}"
        )


# ============================================================
# SUMMARY
# ============================================================

print("\n====================================")
print(
    f"Canonical datasets: "
    f"{len(canonical_data)} / {len(feature_data)}"
)
print("====================================")


if canonical_errors:

    print("\n⚠️ Canonical calculation errors:")

    for symbol, error in canonical_errors.items():

        print(
            f"{symbol}: {error}"
        )


# ============================================================
# VALIDATION
# ============================================================

if canonical_data:

    sample_symbol = next(iter(canonical_data))

    sample = canonical_data[sample_symbol]

    print(
        f"\n🔎 Canonical validation: "
        f"{sample_symbol}"
    )

    display(
        sample[
            [
                "Close",

                "EMA5",
                "EMA10",
                "EMA75",

                "MACD",
                "MACD_SIGNAL",
                "MACD_HIST",

                "STOCH_K",
                "STOCH_D",

                "W_EMA5",
                "W_EMA10",

                "W_MACD",
                "W_MACD_SIGNAL",
                "W_MACD_HIST",

                "VOLUME_RATIO",

                "SQUEEZE",
                "SQUEEZE_RELEASE",

                "CANONICAL_MACD_BULL",
                "CANONICAL_STOCH_BULL",
                "CANONICAL_TREND_BULL",
                "CANONICAL_EARLY_ENTRY",
                "CANONICAL_VOLUME_CONFIRM",

                "CANONICAL_SCORE",
                "CANONICAL_SIGNAL",

                "CANONICAL_WEEKLY_TREND",
                "CANONICAL_WEEKLY_MACD",
                "CANONICAL_WEEKLY_CONFIRM",
            ]
        ].tail(10)
    )


# ============================================================
# SCORE DISTRIBUTION
# ============================================================

if canonical_data:

    all_canonical_scores = pd.concat(
        [
            df["CANONICAL_SCORE"]
            for df in canonical_data.values()
        ]
    )

    print("\n📊 Canonical Score Distribution")

    display(
        all_canonical_scores
        .value_counts()
        .sort_index()
        .rename("bars")
        .to_frame()
    )


    signal_count = int(
        (all_canonical_scores >= 3).sum()
    )

    total_bars = len(
        all_canonical_scores
    )

    signal_rate = (
        signal_count / total_bars * 100
        if total_bars > 0
        else 0
    )


    print(
        f"\nBars with Canonical Score >= 3: "
        f"{signal_count:,}"
    )

    print(
        f"Total usable bars: "
        f"{total_bars:,}"
    )

    print(
        f"Canonical signal rate: "
        f"{signal_rate:.2f}%"
    )


# ============================================================
# LEGACY vs CANONICAL QUICK COMPARISON
# ============================================================

if legacy_data and canonical_data:

    legacy_scores = pd.concat(
        [
            df["LEGACY_SCORE"]
            for df in legacy_data.values()
        ]
    )

    canonical_scores = pd.concat(
        [
            df["CANONICAL_SCORE"]
            for df in canonical_data.values()
        ]
    )

    print("\n====================================")
    print("LEGACY vs CANONICAL")
    print("====================================")

    print(
        f"Legacy signal rate:     "
        f"{(legacy_scores >= 3).mean() * 100:.2f}%"
    )

    print(
        f"Canonical signal rate:  "
        f"{(canonical_scores >= 3).mean() * 100:.2f}%"
    )

In [ ]:

# ============================================================
# 9) MONEY MANAGEMENT — FINAL
# ============================================================

INITIAL_CAPITAL = 100_000.0
MAX_RISK_PCT = 0.02
RISK_PER_TRADE = INITIAL_CAPITAL * MAX_RISK_PCT

LOT_SIZE = 100
ATR_STOP_MULT = 1.5
RECENT_LOW_LOOKBACK = 5


print("💰 Money Management Configuration")
print("------------------------------------")
print(f"Initial Capital : {INITIAL_CAPITAL:,.2f} THB")
print(f"Risk / Trade    : {MAX_RISK_PCT * 100:.2f}%")
print(f"Risk Amount     : {RISK_PER_TRADE:,.2f} THB")
print(f"Lot Size        : {LOT_SIZE:,} shares")
print(f"ATR Stop Mult   : {ATR_STOP_MULT:.2f}x")
print(f"Recent Low      : {RECENT_LOW_LOOKBACK} bars")


# ============================================================
# STOP PRICE
# ============================================================

def calculate_stop_price(
    entry_price,
    atr,
    recent_low,
    atr_multiplier=ATR_STOP_MULT,
):
    if pd.isna(entry_price) or entry_price <= 0:
        return np.nan

    if pd.isna(atr) or atr <= 0:
        return np.nan

    if pd.isna(recent_low) or recent_low <= 0:
        return np.nan

    atr_stop = entry_price - atr * atr_multiplier

    # Conservative stop:
    # choose the LOWER of ATR stop and recent low
    stop_price = min(atr_stop, recent_low)

    if stop_price <= 0:
        return np.nan

    if stop_price >= entry_price:
        return np.nan

    return float(stop_price)


# ============================================================
# POSITION SIZE
# ============================================================

def calculate_position_size(
    capital,
    entry_price,
    stop_price,
    risk_pct=MAX_RISK_PCT,
    lot_size=LOT_SIZE,
):
    if pd.isna(capital) or capital <= 0:
        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": np.nan,
            "position_value": 0.0,
            "status": "INVALID_CAPITAL",
        }

    if pd.isna(entry_price) or entry_price <= 0:
        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": np.nan,
            "position_value": 0.0,
            "status": "INVALID_ENTRY",
        }

    if pd.isna(stop_price) or stop_price <= 0:
        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": np.nan,
            "position_value": 0.0,
            "status": "INVALID_STOP",
        }

    if stop_price >= entry_price:
        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": np.nan,
            "position_value": 0.0,
            "status": "STOP_NOT_BELOW_ENTRY",
        }

    max_risk = capital * risk_pct
    risk_per_share = entry_price - stop_price

    if risk_per_share <= 0:
        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": float(risk_per_share),
            "position_value": 0.0,
            "status": "INVALID_RISK_PER_SHARE",
        }

    # --------------------------------------------------------
    # Limit #1: risk-based position size
    # --------------------------------------------------------

    risk_lots = int(
        (max_risk / risk_per_share) // lot_size
    )

    # --------------------------------------------------------
    # Limit #2: capital-based position size
    # --------------------------------------------------------

    capital_lots = int(
        (capital / entry_price) // lot_size
    )

    # Final lots must satisfy BOTH constraints
    lots = min(risk_lots, capital_lots)

    shares = lots * lot_size

    if shares < lot_size:
        if capital_lots < 1:
            status = "INSUFFICIENT_CAPITAL_FOR_ONE_LOT"
        else:
            status = "ONE_LOT_EXCEEDS_RISK"

        return {
            "shares": 0,
            "lots": 0,
            "risk_amount": 0.0,
            "risk_per_share": float(risk_per_share),
            "position_value": 0.0,
            "status": status,
        }

    actual_risk = shares * risk_per_share
    position_value = shares * entry_price

    if capital_lots < risk_lots:
        status = "CAPITAL_LIMITED"
    else:
        status = "OK"

    return {
        "shares": int(shares),
        "lots": int(lots),
        "risk_amount": float(actual_risk),
        "risk_per_share": float(risk_per_share),
        "position_value": float(position_value),
        "status": status,
    }


# ============================================================
# COMPLETE TRADE PLAN
# ============================================================

def build_trade_plan(
    capital,
    entry_price,
    atr,
    recent_low,
    risk_pct=MAX_RISK_PCT,
    atr_multiplier=ATR_STOP_MULT,
    lot_size=LOT_SIZE,
):
    stop_price = calculate_stop_price(
        entry_price,
        atr,
        recent_low,
        atr_multiplier,
    )

    if pd.isna(stop_price):
        return {
            "entry_price": (
                float(entry_price)
                if pd.notna(entry_price)
                else np.nan
            ),
            "stop_price": np.nan,
            "shares": 0,
            "lots": 0,
            "risk_per_share": np.nan,
            "risk_amount": 0.0,
            "position_value": 0.0,
            "status": "INVALID_STOP",
        }

    position = calculate_position_size(
        capital,
        entry_price,
        stop_price,
        risk_pct,
        lot_size,
    )

    return {
        "entry_price": float(entry_price),
        "stop_price": float(stop_price),
        "shares": position["shares"],
        "lots": position["lots"],
        "risk_per_share": position["risk_per_share"],
        "risk_amount": position["risk_amount"],
        "position_value": position["position_value"],
        "status": position["status"],
    }


# ============================================================
# RECENT LOW
# ============================================================

for symbol, df in canonical_data.items():
    df["RECENT_LOW_5"] = (
        df["Low"]
        .rolling(
            window=RECENT_LOW_LOOKBACK,
            min_periods=RECENT_LOW_LOOKBACK,
        )
        .min()
    )


# ============================================================
# VALIDATION
# ============================================================

print("\n====================================")
print("🧪 Money Management Validation")
print("====================================")


# ------------------------------------------------------------
# Test 1: Normal trade
# ------------------------------------------------------------

test_entry = 50.00
test_atr = 1.50
test_recent_low = 47.00

test_plan = build_trade_plan(
    INITIAL_CAPITAL,
    test_entry,
    test_atr,
    test_recent_low,
)

print("\nTest 1: Normal Trade")
print("------------------------------------")

for key, value in test_plan.items():
    if isinstance(value, float):
        print(f"{key:<20}: {value:,.4f}")
    else:
        print(f"{key:<20}: {value}")


# ------------------------------------------------------------
# Test 2: Stop validation
# ------------------------------------------------------------

test_entry = 10.00
test_atr = 0.50
test_recent_low = 9.80

test_plan_2 = build_trade_plan(
    INITIAL_CAPITAL,
    test_entry,
    test_atr,
    test_recent_low,
)

print("\nTest 2: Stop Validation")
print("------------------------------------")
print(f"Entry : {test_plan_2['entry_price']:.4f}")
print(f"Stop  : {test_plan_2['stop_price']:.4f}")

if pd.notna(test_plan_2["stop_price"]):
    print(
        "Valid : "
        f"{test_plan_2['stop_price'] < test_plan_2['entry_price']}"
    )
else:
    print("Valid : False")


# ------------------------------------------------------------
# Test 3: One lot risk check
# ------------------------------------------------------------

test_entry = 80.00
test_stop = 60.00

test_position = calculate_position_size(
    INITIAL_CAPITAL,
    test_entry,
    test_stop,
)

print("\nTest 3: One Lot Risk Check")
print("------------------------------------")
print(f"Entry       : {test_entry:.2f}")
print(f"Stop        : {test_stop:.2f}")
print(f"Risk / Share: {test_position['risk_per_share']:.2f}")
print(f"Shares      : {test_position['shares']:,}")
print(f"Lots        : {test_position['lots']:,}")
print(f"Risk Amount : {test_position['risk_amount']:,.2f}")
print(f"Position    : {test_position['position_value']:,.2f}")
print(f"Status      : {test_position['status']}")


# ------------------------------------------------------------
# Test 4: Capital limit
# ------------------------------------------------------------

test_entry = 1.00
test_stop = 0.99

test_position_4 = calculate_position_size(
    INITIAL_CAPITAL,
    test_entry,
    test_stop,
)

print("\nTest 4: Capital Limit Check")
print("------------------------------------")
print(f"Entry       : {test_entry:.2f}")
print(f"Stop        : {test_stop:.2f}")
print(f"Risk / Share: {test_position_4['risk_per_share']:.4f}")
print(f"Shares      : {test_position_4['shares']:,}")
print(f"Lots        : {test_position_4['lots']:,}")
print(f"Risk Amount : {test_position_4['risk_amount']:,.2f}")
print(f"Position    : {test_position_4['position_value']:,.2f}")
print(f"Status      : {test_position_4['status']}")

assert (
    test_position_4["position_value"]
    <= INITIAL_CAPITAL + 1e-9
), "Position value exceeds capital"


# ============================================================
# REAL DATA SANITY CHECK
# ============================================================

print("\n====================================")
print("🔎 Real Data Sanity Check")
print("====================================")

if canonical_data:
    sample_symbol = next(iter(canonical_data))
    sample = canonical_data[sample_symbol]

    valid_rows = (
        sample[
            ["Close", "ATR14", "RECENT_LOW_5"]
        ]
        .dropna()
        .tail(3)
    )

    print(f"\nSample: {sample_symbol}")
    display(valid_rows)


# ============================================================
# FINAL CHECKS
# ============================================================

assert INITIAL_CAPITAL > 0
assert 0 < MAX_RISK_PCT <= 0.05
assert LOT_SIZE > 0
assert ATR_STOP_MULT > 0
assert RECENT_LOW_LOOKBACK > 0

print("\n====================================")
print("✅ Money Management Cell Ready")
print("====================================")
print(f"Maximum risk per trade: {RISK_PER_TRADE:,.2f} THB")
print("Stop rule: min(Entry - ATR × 1.5, Recent Low)")
print("Position size: whole 100-share lots, rounded DOWN")
print("Position limit: Risk limit + Capital limit")
print("Target prices: calculated later from actual Entry")

In [ ]:

# ============================================================
# 10) CURRENT SCANNER
# ============================================================

print("🔎 CURRENT SCANNER")
print("=" * 90)


def build_current_scan_row(symbol, df):
    if df is None or df.empty:
        return {
            "Symbol": symbol,
            "Date": pd.NaT,
            "Close": np.nan,
            "Score": np.nan,
            "Signal": False,
            "WeeklyConfirm": False,
            "MACD": np.nan,
            "MACD_Signal": np.nan,
            "MACD_Hist": np.nan,
            "EMA5": np.nan,
            "EMA10": np.nan,
            "EMA75": np.nan,
            "W_EMA5": np.nan,
            "W_EMA10": np.nan,
            "W_MACD": np.nan,
            "W_MACD_Signal": np.nan,
            "ATR14": np.nan,
            "RecentLow5": np.nan,
            "Stop": np.nan,
            "RiskPerShare": np.nan,
            "Shares": 0,
            "Lots": 0,
            "RiskAmount": 0.0,
            "PositionValue": 0.0,
            "Target1": np.nan,
            "Target2": np.nan,
            "Status": "NO_DATA",
        }

    row = df.iloc[-1]

    entry_price = row["Close"]
    atr = row["ATR14"]
    recent_low = row["RECENT_LOW_5"]

    trade_plan = build_trade_plan(
        INITIAL_CAPITAL,
        entry_price,
        atr,
        recent_low,
        atr_multiplier=PRODUCTION_ATR_MULT,
    )

    score = row.get("CANONICAL_SCORE", np.nan)
    signal = (
        symbol in PRODUCTION_SYMBOLS
        and float(row.get("CANONICAL_SCORE", 0)) >= PRODUCTION_MIN_SCORE
        and (
            not PRODUCTION_WEEKLY_CONFIRM
            or bool(row.get("CANONICAL_WEEKLY_CONFIRM", False))
        )
    )

    weekly_trend = bool(
        row.get("CANONICAL_WEEKLY_TREND", False)
    )

    weekly_macd = bool(
        row.get("CANONICAL_WEEKLY_MACD", False)
    )

    weekly_confirm = bool(
        row.get("CANONICAL_WEEKLY_CONFIRM", False)
    )

    target1 = (
        float(entry_price * 1.07)
        if pd.notna(entry_price)
        else np.nan
    )

    target2 = (
        float(entry_price * (1 + PRODUCTION_TARGET2_PCT))
        if pd.notna(entry_price)
        else np.nan
    )

    return {
        "Symbol": symbol,
        "Date": df.index[-1],
        "Close": float(entry_price),
        "Score": score,
        "Signal": signal,
        "WeeklyTrend": weekly_trend,
        "WeeklyMACD": weekly_macd,
        "WeeklyConfirm": weekly_confirm,
        "MACD": row["MACD"],
        "MACD_Signal": row["MACD_SIGNAL"],
        "MACD_Hist": row["MACD_HIST"],
        "EMA5": row["EMA5"],
        "EMA10": row["EMA10"],
        "EMA75": row["EMA75"],
        "W_EMA5": row["W_EMA5"],
        "W_EMA10": row["W_EMA10"],
        "W_MACD": row["W_MACD"],
        "W_MACD_Signal": row["W_MACD_SIGNAL"],
        "ATR14": row["ATR14"],
        "RecentLow5": row["RECENT_LOW_5"],
        "Stop": trade_plan["stop_price"],
        "RiskPerShare": trade_plan["risk_per_share"],
        "Shares": trade_plan["shares"],
        "Lots": trade_plan["lots"],
        "RiskAmount": trade_plan["risk_amount"],
        "PositionValue": trade_plan["position_value"],
        "Target1": target1,
        "Target2": target2,
        "Status": trade_plan["status"],
    }


# ============================================================
# BUILD CURRENT SCAN
# ============================================================

current_scan_rows = []

for symbol, df in canonical_data.items():
    current_scan_rows.append(
        build_current_scan_row(symbol, df)
    )

current_scan = pd.DataFrame(current_scan_rows)


# ============================================================
# SORT
# ============================================================

current_scan = current_scan.sort_values(
    by=["Signal", "Score", "WeeklyConfirm"],
    ascending=[False, False, False],
    na_position="last",
).reset_index(drop=True)


# ============================================================
# DISPLAY
# ============================================================

print("\n📊 Current Market Snapshot")
print("-" * 90)

display(
    current_scan[
        [
            "Symbol",
            "Date",
            "Close",
            "Score",
            "Signal",
            "WeeklyConfirm",
            "ATR14",
            "Stop",
            "Shares",
            "RiskAmount",
            "Target1",
            "Target2",
            "Status",
        ]
    ]
)


# ============================================================
# SIGNAL SUMMARY
# ============================================================

signal_scan = current_scan[
    current_scan["Signal"] == True
].copy()

weekly_confirmed = current_scan[
    current_scan["WeeklyConfirm"] == True
].copy()

print("\n====================================")
print("📈 CURRENT SCANNER SUMMARY")
print("====================================")

print(f"Total symbols       : {len(current_scan)}")
print(f"Production universe : {len(PRODUCTION_SYMBOLS)}")
print(f"Production min score: {PRODUCTION_MIN_SCORE}")
print(f"Production ATR mult : {PRODUCTION_ATR_MULT:.2f}x")
print(f"Production target2  : {PRODUCTION_TARGET2_PCT * 100:.1f}%")
print(f"Canonical signals   : {len(signal_scan)}")
print(f"Weekly confirmed    : {len(weekly_confirmed)}")

if len(current_scan) > 0:
    signal_rate = (
        len(signal_scan) / len(current_scan) * 100
    )
else:
    signal_rate = 0.0

print(f"Signal rate         : {signal_rate:.2f}%")


# ============================================================
# TOP SIGNALS
# ============================================================

print("\n====================================")
print("🏆 TOP CURRENT SIGNALS")
print("====================================")

if signal_scan.empty:
    print("No current Canonical signals.")
else:
    display(
        signal_scan[
            [
                "Symbol",
                "Date",
                "Close",
                "Score",
                "WeeklyConfirm",
                "ATR14",
                "Stop",
                "Shares",
                "RiskAmount",
                "Target1",
                "Target2",
            ]
        ].head(10)
    )


# ============================================================
# SANITY CHECKS
# ============================================================

assert len(current_scan) == len(canonical_data)

assert current_scan["Score"].dropna().between(0, 5).all()

assert (
    current_scan["RiskAmount"].fillna(0)
    <= RISK_PER_TRADE + 1e-9
).all()

assert (
    current_scan["PositionValue"].fillna(0)
    <= INITIAL_CAPITAL + 1e-9
).all()

print("\n====================================")
print("✅ Current Scanner Cell Ready")
print("====================================")

In [ ]:
# ============================================================
# 10A) PRODUCTION SIGNAL JOURNAL
# ============================================================
#
# Audit-only persistence for the frozen Production scanner.
# This cell does NOT change signal logic, thresholds, or sizing rules.
# ReferenceEntryPrice is the signal-day Close used only as a planning reference.
# It is NOT an executed entry price and does not use future data.
# ============================================================

from pathlib import Path

JOURNAL_PATH = Path("/content/production_signal_journal.csv")

JOURNAL_COLUMNS = [
    "RunTimestampUTC",
    "ScanDate",
    "Symbol",
    "Score",
    "Signal",
    "WeeklyConfirm",
    "SignalClose",
    "ReferenceEntryPrice",
    "ATR14",
    "RecentLow5",
    "StopPrice",
    "Target1",
    "Target2",
    "PlannedShares",
    "PlannedLots",
    "PlannedRiskAmount",
    "PlannedPositionValue",
    "Status",
    "ProductionMinScore",
    "ProductionWeeklyConfirm",
    "ProductionATRMult",
    "ProductionTarget2Pct",
    "ProductionMaxHoldingDays",
]

journal = current_scan.copy()
journal = journal[journal["Symbol"].isin(PRODUCTION_SYMBOLS)].copy()

run_timestamp = pd.Timestamp.now(tz="UTC").isoformat()
journal["RunTimestampUTC"] = run_timestamp
journal["ScanDate"] = pd.to_datetime(journal["Date"]).dt.strftime("%Y-%m-%d")
journal["ReferenceEntryPrice"] = journal["Close"]
journal["PlannedShares"] = journal["Shares"].fillna(0).astype(int)
journal["PlannedLots"] = journal["Lots"].fillna(0).astype(int)
journal["PlannedRiskAmount"] = journal["RiskAmount"].fillna(0.0)
journal["PlannedPositionValue"] = journal["PositionValue"].fillna(0.0)

journal = journal.rename(columns={
    "Close": "SignalClose",
    "Stop": "StopPrice",
})

journal["ProductionMinScore"] = PRODUCTION_MIN_SCORE
journal["ProductionWeeklyConfirm"] = PRODUCTION_WEEKLY_CONFIRM
journal["ProductionATRMult"] = PRODUCTION_ATR_MULT
journal["ProductionTarget2Pct"] = PRODUCTION_TARGET2_PCT
journal["ProductionMaxHoldingDays"] = PRODUCTION_MAX_HOLDING_DAYS

journal = journal[JOURNAL_COLUMNS].copy()

if JOURNAL_PATH.exists():
    existing = pd.read_csv(JOURNAL_PATH)
    journal = pd.concat([existing, journal], ignore_index=True)

journal = journal.drop_duplicates(
    subset=["ScanDate", "Symbol"],
    keep="last",
).sort_values(
    ["ScanDate", "Symbol"]
).reset_index(drop=True)

JOURNAL_PATH.parent.mkdir(parents=True, exist_ok=True)
journal.to_csv(JOURNAL_PATH, index=False)

print("\n====================================")
print("📝 PRODUCTION SIGNAL JOURNAL")
print("====================================")
print(f"Journal path      : {JOURNAL_PATH}")
print(f"Production symbols: {len(PRODUCTION_SYMBOLS)}")
print(f"Rows stored       : {len(journal)}")
print(
    f"Latest scan date  : "
    f"{journal['ScanDate'].max() if not journal.empty else 'N/A'}"
)
print(
    f"Current signals   : "
    f"{int(journal[journal['ScanDate'] == journal['ScanDate'].max()]['Signal'].sum()) if not journal.empty else 0}"
)
print("Actual execution  : not recorded until a real fill is available")

assert set(PRODUCTION_SYMBOLS) == set(
    journal[
        journal["ScanDate"] == journal["ScanDate"].max()
    ]["Symbol"]
)
assert not journal.duplicated(
    ["ScanDate", "Symbol"]
).any()

print("JOURNAL_VALID")


In [ ]:

# ============================================================
# 11) HISTORICAL SIGNAL GENERATION
# ============================================================

print("📡 HISTORICAL SIGNAL GENERATION")
print("=" * 90)


SIGNAL_MIN_SCORE = 3


def generate_historical_signals(
    symbol,
    df,
    min_score=SIGNAL_MIN_SCORE,
):
    if df is None or df.empty:
        return pd.DataFrame()

    data = df.copy()

    # --------------------------------------------------------
    # Required columns
    # --------------------------------------------------------

    required = [
        "Close",
        "Open",
        "High",
        "Low",
        "ATR14",
        "RECENT_LOW_5",
        "CANONICAL_SCORE",
        "CANONICAL_SIGNAL",
        "CANONICAL_WEEKLY_TREND",
        "CANONICAL_WEEKLY_MACD",
        "CANONICAL_WEEKLY_CONFIRM",
    ]

    missing = [
        col for col in required
        if col not in data.columns
    ]

    if missing:
        raise ValueError(
            f"{symbol}: missing columns {missing}"
        )

    # --------------------------------------------------------
    # Signal rows
    # --------------------------------------------------------

    signal_mask = (
        data["CANONICAL_SIGNAL"].fillna(False)
        & (data["CANONICAL_SCORE"] >= min_score)
    )

    signal_dates = data.index[signal_mask]

    rows = []

    for signal_date in signal_dates:
        signal_pos = data.index.get_loc(signal_date)

        # Need next trading day for entry
        if signal_pos + 1 >= len(data):
            continue

        entry_date = data.index[signal_pos + 1]

        signal_row = data.iloc[signal_pos]
        entry_row = data.iloc[signal_pos + 1]

        signal_close = signal_row["Close"]
        signal_atr = signal_row["ATR14"]
        recent_low = signal_row["RECENT_LOW_5"]

        # ----------------------------------------------------
        # Stop is calculated from information available
        # at SIGNAL DATE.
        # ----------------------------------------------------

        stop_price = calculate_stop_price(
            signal_close,
            signal_atr,
            recent_low,
        )

        # ----------------------------------------------------
        # Entry is NEXT DAY OPEN.
        # ----------------------------------------------------

        entry_price = entry_row["Open"]

        # ----------------------------------------------------
        # If entry is invalid, skip.
        # ----------------------------------------------------

        if pd.isna(entry_price) or entry_price <= 0:
            continue

        if pd.isna(stop_price) or stop_price <= 0:
            continue

        # ----------------------------------------------------
        # Position sizing uses actual next-day entry.
        # ----------------------------------------------------

        trade_plan = build_trade_plan(
            INITIAL_CAPITAL,
            entry_price,
            signal_atr,
            recent_low,
        )

        # ----------------------------------------------------
        # Targets based on ACTUAL entry.
        # ----------------------------------------------------

        target1 = entry_price * 1.07
        target2 = entry_price * 1.15

        rows.append({
            "Symbol": symbol,

            "SignalDate": signal_date,
            "EntryDate": entry_date,

            "SignalClose": float(signal_close),
            "EntryPrice": float(entry_price),

            "Score": int(signal_row["CANONICAL_SCORE"]),

            "WeeklyTrend": bool(
                signal_row["CANONICAL_WEEKLY_TREND"]
            ),

            "WeeklyMACD": bool(
                signal_row["CANONICAL_WEEKLY_MACD"]
            ),

            "WeeklyConfirm": bool(
                signal_row["CANONICAL_WEEKLY_CONFIRM"]
            ),

            "ATR14": float(signal_atr),

            "RecentLow5": float(recent_low),

            "StopPrice": float(
                trade_plan["stop_price"]
            ),

            "RiskPerShare": float(
                trade_plan["risk_per_share"]
            ),

            "Shares": int(
                trade_plan["shares"]
            ),

            "Lots": int(
                trade_plan["lots"]
            ),

            "RiskAmount": float(
                trade_plan["risk_amount"]
            ),

            "PositionValue": float(
                trade_plan["position_value"]
            ),

            "Target1": float(target1),
            "Target2": float(target2),

            "EntryOpen": float(entry_row["Open"]),
            "EntryHigh": float(entry_row["High"]),
            "EntryLow": float(entry_row["Low"]),
            "EntryClose": float(entry_row["Close"]),
        })

    return pd.DataFrame(rows)


# ============================================================
# GENERATE SIGNALS FOR ALL SYMBOLS
# ============================================================

historical_signals_list = []

for symbol, df in canonical_data.items():

    signals = generate_historical_signals(
        symbol,
        df,
        min_score=SIGNAL_MIN_SCORE,
    )

    if not signals.empty:
        historical_signals_list.append(signals)


if historical_signals_list:
    historical_signals = pd.concat(
        historical_signals_list,
        ignore_index=True,
    )
else:
    historical_signals = pd.DataFrame()


# ============================================================
# SORT
# ============================================================

if not historical_signals.empty:
    historical_signals = historical_signals.sort_values(
        ["SignalDate", "Score"],
        ascending=[True, False],
    ).reset_index(drop=True)


# ============================================================
# SUMMARY
# ============================================================

print("\n====================================")
print("📊 HISTORICAL SIGNAL SUMMARY")
print("====================================")

print(
    f"Total signals : {len(historical_signals):,}"
)

if not historical_signals.empty:

    print(
        f"Symbols       : "
        f"{historical_signals['Symbol'].nunique()}"
    )

    print(
        f"Score >= 3    : "
        f"{(historical_signals['Score'] >= 3).sum():,}"
    )

    print(
        f"Weekly confirm: "
        f"{historical_signals['WeeklyConfirm'].sum():,}"
    )

    print(
        f"Date range    : "
        f"{historical_signals['SignalDate'].min().date()} "
        f"→ "
        f"{historical_signals['SignalDate'].max().date()}"
    )


# ============================================================
# LOOKAHEAD SANITY CHECK
# ============================================================

print("\n====================================")
print("🔬 LOOKAHEAD SANITY CHECK")
print("====================================")

if not historical_signals.empty:

    invalid_entry = (
        historical_signals["EntryDate"]
        <= historical_signals["SignalDate"]
    ).sum()

    invalid_price = (
        historical_signals["EntryPrice"] <= 0
    ).sum()

    invalid_stop = (
        historical_signals["StopPrice"]
        >= historical_signals["SignalClose"]
    ).sum()

    print(
        f"EntryDate <= SignalDate : {invalid_entry}"
    )

    print(
        f"Invalid EntryPrice      : {invalid_price}"
    )

    print(
        f"Invalid Stop             : {invalid_stop}"
    )

    assert invalid_entry == 0
    assert invalid_price == 0
    assert invalid_stop == 0


# ============================================================
# SAMPLE SIGNALS
# ============================================================

print("\n====================================")
print("🔎 SAMPLE HISTORICAL SIGNALS")
print("====================================")

if historical_signals.empty:
    print("No historical signals found.")
else:
    display(
        historical_signals[
            [
                "Symbol",
                "SignalDate",
                "EntryDate",
                "SignalClose",
                "EntryPrice",
                "Score",
                "WeeklyConfirm",
                "ATR14",
                "StopPrice",
                "Shares",
                "RiskAmount",
                "Target1",
                "Target2",
            ]
        ].tail(15)
    )


# ============================================================
# FINAL CHECK
# ============================================================

print("\n====================================")
print("✅ Historical Signal Generation Ready")
print("====================================")

In [ ]:

# ============================================================
# 11.2) FIX HISTORICAL SIGNALS
# PRESERVE ORIGINAL SIGNAL SET
# FIX STOP FROM SIGNAL-DATE DATA
# ============================================================

print("🔧 FIXING HISTORICAL SIGNALS")
print("=" * 90)

old_signals = historical_signals.copy()

print(f"Original signals : {len(old_signals):,}")

fixed_rows = []
excluded_invalid_stop = 0

for _, old in old_signals.iterrows():

    symbol = old["Symbol"]
    signal_date = pd.Timestamp(old["SignalDate"])
    entry_date = pd.Timestamp(old["EntryDate"])

    if symbol not in feature_data:
        continue

    df = feature_data[symbol].sort_index()

    if signal_date not in df.index:
        continue

    if entry_date not in df.index:
        continue

    # RECENT_LOW_5 is created on canonical_data in Cell 9.
    # feature_data may not carry that derived column, so rebuild it
    # here from signal-date history if it is missing.
    if "RECENT_LOW_5" not in df.columns:
        df = df.copy()
        df["RECENT_LOW_5"] = (
            df["Low"]
            .rolling(
                window=RECENT_LOW_LOOKBACK,
                min_periods=RECENT_LOW_LOOKBACK,
            )
            .min()
        )

    signal_row = df.loc[signal_date]
    entry_row = df.loc[entry_date]

    signal_close = float(signal_row["Close"])
    signal_atr = float(signal_row["ATR14"])
    recent_low_value = signal_row["RECENT_LOW_5"]

    if not np.isfinite(recent_low_value) or recent_low_value <= 0:
        continue

    recent_low = float(recent_low_value)
    entry_price = float(entry_row["Open"])

    # --------------------------------------------------------
    # FIXED STOP FROM SIGNAL DATE
    # --------------------------------------------------------

    stop_price = calculate_stop_price(
        entry_price=signal_close,
        atr=signal_atr,
        recent_low=recent_low,
    )

    if not np.isfinite(stop_price) or stop_price <= 0:
        excluded_invalid_stop += 1
        continue

    # A long trade cannot have its protective stop at or above
    # the actual next-day entry price. This can happen after a
    # gap-down between signal date and entry date. Exclude that
    # trade from the executable historical set rather than
    # manufacturing a stop or silently changing the strategy.
    if entry_price <= stop_price:
        excluded_invalid_stop += 1
        continue

    # --------------------------------------------------------
    # POSITION SIZE
    # Uses actual entry price + fixed stop
    # --------------------------------------------------------

    risk_per_share = entry_price - stop_price

    if risk_per_share <= 0:
        shares = 0
        lots = 0
        risk_amount = 0.0
        position_value = 0.0
        status = "INVALID_STOP"

    else:
        max_risk = INITIAL_CAPITAL * MAX_RISK_PCT

        risk_shares = int(max_risk / risk_per_share)
        capital_shares = int(INITIAL_CAPITAL / entry_price)

        shares = min(risk_shares, capital_shares)

        shares = (shares // LOT_SIZE) * LOT_SIZE
        lots = shares // LOT_SIZE

        risk_amount = risk_per_share * shares
        position_value = entry_price * shares

        status = "OK" if shares > 0 else "CAPITAL_LIMITED"

    # --------------------------------------------------------
    # TARGETS
    # --------------------------------------------------------

    target1 = entry_price * 1.07
    target2 = entry_price * 1.15

    fixed_rows.append({
        "Symbol": symbol,
        "SignalDate": signal_date,
        "EntryDate": entry_date,
        "SignalClose": signal_close,
        "EntryPrice": entry_price,
        "Score": int(old["Score"]),
        "WeeklyConfirm": bool(old["WeeklyConfirm"]),
        "ATR14": signal_atr,
        "RecentLow5": recent_low,
        "StopPrice": stop_price,
        "Shares": shares,
        "Lots": lots,
        "RiskAmount": risk_amount,
        "PositionValue": position_value,
        "Target1": target1,
        "Target2": target2,
        "Status": status,
        "EntryOpen": float(entry_row["Open"]),
        "EntryHigh": float(entry_row["High"]),
        "EntryLow": float(entry_row["Low"]),
        "EntryClose": float(entry_row["Close"]),
    })


# ------------------------------------------------------------
# SAFETY CHECK
# ------------------------------------------------------------

if not fixed_rows:
    raise RuntimeError(
        "No signals were rebuilt. Stop here and inspect Cell 11."
    )


historical_signals = pd.DataFrame(fixed_rows)

historical_signals = (
    historical_signals
    .sort_values(["SignalDate", "Symbol"])
    .reset_index(drop=True)
)


# ============================================================
# SUMMARY
# ============================================================

print("\n====================================")
print("📊 REBUILD SUMMARY")
print("====================================")

print(f"Original signals : {len(old_signals):,}")
print(f"Fixed signals    : {len(historical_signals):,}")
print(
    f"Difference       : "
    f"{len(historical_signals) - len(old_signals):+,}"
)


# ============================================================
# VALIDATION
# ============================================================

entry_before_signal = (
    historical_signals["EntryDate"]
    < historical_signals["SignalDate"]
).sum()

invalid_entry = (
    ~np.isfinite(historical_signals["EntryPrice"])
    | (historical_signals["EntryPrice"] <= 0)
).sum()

invalid_stop = (
    ~np.isfinite(historical_signals["StopPrice"])
    | (historical_signals["StopPrice"] <= 0)
).sum()

stop_ge_entry = (
    historical_signals["StopPrice"]
    >= historical_signals["EntryPrice"]
).sum()

risk_over_limit = (
    historical_signals["RiskAmount"]
    > INITIAL_CAPITAL * MAX_RISK_PCT + 1e-9
).sum()

capital_over_limit = (
    historical_signals["PositionValue"]
    > INITIAL_CAPITAL + 1e-9
).sum()


print("\n====================================")
print("🔍 VALIDATION")
print("====================================")

print(f"Entry before Signal : {entry_before_signal:,}")
print(f"Invalid EntryPrice  : {invalid_entry:,}")
print(f"Invalid Stop        : {invalid_stop:,}")
print(f"Stop >= Entry       : {stop_ge_entry:,}")
print(f"Risk > 2%           : {risk_over_limit:,}")
print(f"Capital > 100k      : {capital_over_limit:,}")


# ============================================================
# STOP CONSISTENCY
# ============================================================

check = historical_signals.head(10).copy()

check["ExpectedStop"] = check.apply(
    lambda r: calculate_stop_price(
        r["SignalClose"],
        r["ATR14"],
        r["RecentLow5"],
    ),
    axis=1,
)

check["StopMatch"] = np.isclose(
    check["StopPrice"],
    check["ExpectedStop"],
    rtol=1e-10,
    atol=1e-10,
)

print("\n====================================")
print("🔒 STOP CONSISTENCY")
print("====================================")

display(
    check[
        [
            "Symbol",
            "SignalDate",
            "SignalClose",
            "EntryPrice",
            "ATR14",
            "RecentLow5",
            "StopPrice",
            "ExpectedStop",
            "StopMatch",
        ]
    ]
)


# ============================================================
# FINAL STATUS
# ============================================================

print("\n====================================")

assert len(historical_signals) + excluded_invalid_stop == len(old_signals), (
    "Signal accounting mismatch during rebuild."
)
assert entry_before_signal == 0, "EntryDate is before SignalDate."
assert invalid_entry == 0, "Invalid EntryPrice found."
assert invalid_stop == 0, "Invalid StopPrice found."
assert stop_ge_entry == 0, "StopPrice >= EntryPrice found."
assert risk_over_limit == 0, "Risk exceeds configured limit."
assert capital_over_limit == 0, "Position value exceeds initial capital."
assert check["StopMatch"].all(), "Stop-price consistency check failed."

print("✅ HISTORICAL SIGNALS FIXED")
print("CELL 11.2 PASSED")
print("====================================")

In [ ]:

# ============================================================
# 12.1) SIGNAL CLUSTER ANALYSIS
# NO TRADE-ENGINE DEPENDENCY
# ============================================================

print("🔍 SIGNAL CLUSTER ANALYSIS")
print("=" * 90)

assert not historical_signals.empty, "historical_signals is empty."

analysis = historical_signals.copy()
analysis["SignalDate"] = pd.to_datetime(analysis["SignalDate"])
analysis["EntryDate"] = pd.to_datetime(analysis["EntryDate"])

analysis = analysis.sort_values(
    ["Symbol", "SignalDate"]
).reset_index(drop=True)

# ------------------------------------------------------------
# Days since previous signal for the same symbol
# ------------------------------------------------------------

analysis["PrevSignalDate"] = (
    analysis.groupby("Symbol")["SignalDate"].shift(1)
)

analysis["DaysSincePrevSignal"] = (
    analysis["SignalDate"] - analysis["PrevSignalDate"]
).dt.days

analysis["ClusteredSignal"] = (
    analysis["DaysSincePrevSignal"].notna()
    & (analysis["DaysSincePrevSignal"] <= 3)
)

clustered = analysis[analysis["ClusteredSignal"]].copy()

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

total_signals = len(analysis)
clustered_count = len(clustered)
cluster_rate = clustered_count / total_signals * 100

print("\n====================================")
print("📊 SIGNAL CLUSTER SUMMARY")
print("====================================")
print(f"Total signals     : {total_signals:,}")
print(f"Clustered signals : {clustered_count:,}")
print(f"Cluster rate      : {cluster_rate:.2f}%")

# ------------------------------------------------------------
# Symbol-level summary
# ------------------------------------------------------------

symbol_cluster = (
    analysis.groupby("Symbol")
    .agg(
        Signals=("SignalDate", "count"),
        Clustered=("ClusteredSignal", "sum"),
    )
)

symbol_cluster["ClusterRate"] = (
    symbol_cluster["Clustered"]
    / symbol_cluster["Signals"]
    * 100
)

symbol_cluster = symbol_cluster.sort_values(
    ["Signals", "ClusterRate"],
    ascending=[False, False],
)

print("\n====================================")
print("📈 SIGNALS BY SYMBOL")
print("====================================")
display(symbol_cluster)

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

same_symbol_order_ok = (
    analysis.groupby("Symbol")["SignalDate"]
    .apply(lambda s: s.is_monotonic_increasing)
    .all()
)

invalid_gap = (
    analysis["DaysSincePrevSignal"].notna()
    & (analysis["DaysSincePrevSignal"] < 0)
).sum()

assert same_symbol_order_ok, "Signal dates are not chronological by symbol."
assert invalid_gap == 0, "Negative signal-date gap found."
assert clustered_count <= total_signals

print("\n====================================")
print("🔎 RECENT CLUSTERED SIGNALS")
print("====================================")

if clustered.empty:
    print("No clustered signals found.")
else:
    display(
        clustered[
            [
                "Symbol",
                "SignalDate",
                "EntryDate",
                "Score",
                "WeeklyConfirm",
                "DaysSincePrevSignal",
            ]
        ].tail(30)
    )

print("\n====================================")
print("✅ CELL 12.1 PASSED")
print("====================================")

In [ ]:
# ============================================================
# CELL 12.2 — NON-OVERLAPPING BACKTEST
# ============================================================

print("=" * 90)
print("🔬 NON-OVERLAPPING HISTORICAL BACKTEST")
print("=" * 90)

# ------------------------------------------------------------
# 1) Conservative trade simulator
# ------------------------------------------------------------

def simulate_trade_fixed_stop(signal, price_data):
    """
    Simulate one long trade with:
      - next-day entry
      - conservative gap/stop handling
      - target 1 tracking
      - target 2 exit
      - hard MAX_HOLDING_DAYS time exit
      - MAE/MFE measured only while the trade is actually open

    MAX_HOLDING_DAYS is interpreted as calendar days from EntryDate.
    On the maximum-holding-date bar, STOP/TARGET rules have priority;
    otherwise the position is closed at that day's Close.
    """

    entry_date = pd.Timestamp(signal["EntryDate"])
    entry_price = float(signal["EntryPrice"])
    stop_price = float(signal["StopPrice"])
    shares = int(signal["Shares"])

    if entry_price <= 0 or stop_price <= 0 or shares <= 0:
        return None

    target1 = entry_price * 1.07
    target2 = entry_price * 1.15

    future = price_data.loc[
        price_data.index >= entry_date
    ].copy()

    if future.empty:
        return None

    max_hold_date = entry_date + pd.Timedelta(
        days=MAX_HOLDING_DAYS
    )

    future = future.loc[
        future.index <= max_hold_date
    ].copy()

    if future.empty:
        return None

    target1_hit = False
    exit_date = pd.NaT
    exit_price = np.nan
    exit_reason = None

    for current_date, bar in future.iterrows():

        day_open = float(bar["Open"])
        day_high = float(bar["High"])
        day_low = float(bar["Low"])
        day_close = float(bar["Close"])

        # ----------------------------------------------------
        # Gap handling
        # ----------------------------------------------------
        # If market opens below stop, actual exit is OPEN.
        # This is conservative and avoids assuming execution
        # at a better price than the market provided.
        if day_open <= stop_price:
            exit_date = current_date
            exit_price = day_open
            exit_reason = "STOP_GAP"
            break

        # ----------------------------------------------------
        # Normal intraday stop
        # ----------------------------------------------------
        if day_low <= stop_price:
            exit_date = current_date
            exit_price = stop_price
            exit_reason = "STOP"
            break

        # ----------------------------------------------------
        # Target 2
        # ----------------------------------------------------
        if day_open >= target2:
            target1_hit = True
            exit_date = current_date
            exit_price = day_open
            exit_reason = "TARGET2_GAP"
            break

        if day_high >= target2:
            target1_hit = True
            exit_date = current_date
            exit_price = target2
            exit_reason = "TARGET2"
            break

        # ----------------------------------------------------
        # Target 1 — tracking only
        # ----------------------------------------------------
        if day_high >= target1:
            target1_hit = True

        # ----------------------------------------------------
        # Hard time exit
        # ----------------------------------------------------
        # STOP/TARGET checks above always have priority on this
        # date. If neither occurs, exit at the closing price.
        if current_date >= max_hold_date:
            exit_date = current_date
            exit_price = day_close
            exit_reason = "MAX_HOLD"
            break

    # --------------------------------------------------------
    # End of data
    # --------------------------------------------------------

    if pd.isna(exit_date):
        # This can only happen when the available dataset ends
        # before MAX_HOLDING_DAYS. Close at the final available bar.
        exit_date = future.index[-1]
        exit_price = float(future.iloc[-1]["Close"])
        exit_reason = "EOD"

    holding_days = (
        pd.Timestamp(exit_date) - entry_date
    ).days

    if holding_days > MAX_HOLDING_DAYS:
        raise AssertionError(
            f"MAX_HOLDING_DAYS violated: "
            f"{holding_days} > {MAX_HOLDING_DAYS}"
        )

    pnl = (exit_price - entry_price) * shares
    return_pct = (
        (exit_price - entry_price)
        / entry_price
        * 100
    )

    # IMPORTANT:
    # MAE/MFE must be calculated only over the period in which
    # the position was actually open. Using all future bars would
    # introduce look-ahead leakage into the diagnostics.
    held_future = price_data.loc[
        (price_data.index >= entry_date)
        & (price_data.index <= exit_date)
    ].copy()

    min_low = float(held_future["Low"].min())
    max_high = float(held_future["High"].max())

    mae_pct = (
        (min_low - entry_price)
        / entry_price
        * 100
    )

    mfe_pct = (
        (max_high - entry_price)
        / entry_price
        * 100
    )

    return {
        "Symbol": signal["Symbol"],
        "SignalDate": signal["SignalDate"],
        "EntryDate": entry_date,
        "EntryPrice": entry_price,
        "StopPrice": stop_price,
        "Shares": shares,
        "Target1": target1,
        "Target2": target2,
        "Target1Hit": target1_hit,
        "ExitDate": exit_date,
        "ExitPrice": exit_price,
        "ExitReason": exit_reason,
        "HoldingDays": holding_days,
        "PnL": pnl,
        "ReturnPct": return_pct,
        "MAE_Pct": mae_pct,
        "MFE_Pct": mfe_pct,
        "Score": signal["Score"],
        "WeeklyConfirm": signal["WeeklyConfirm"],
    }


# ------------------------------------------------------------
# 2) Backtest one symbol without overlapping trades
# ------------------------------------------------------------

def backtest_one_symbol_nonoverlap(symbol):

    signals = historical_signals[
        historical_signals["Symbol"] == symbol
    ].copy()

    signals = signals.sort_values(
        ["EntryDate", "SignalDate"]
    ).reset_index(drop=True)

    if symbol not in feature_data:
        return pd.DataFrame()

    price_data = feature_data[symbol].copy()

    trades = []
    previous_exit_date = pd.NaT
    skipped_overlap = 0

    for _, signal in signals.iterrows():

        entry_date = pd.Timestamp(signal["EntryDate"])

        # ----------------------------------------------------
        # Prevent overlapping trades
        # ----------------------------------------------------
        if not pd.isna(previous_exit_date):
            if entry_date <= previous_exit_date:
                skipped_overlap += 1
                continue

        trade = simulate_trade_fixed_stop(
            signal,
            price_data
        )

        if trade is None:
            continue

        trades.append(trade)
        previous_exit_date = trade["ExitDate"]

    result = pd.DataFrame(trades)

    return result


# ------------------------------------------------------------
# 3) Test AU.BK first
# ------------------------------------------------------------

TEST_SYMBOL = WATCHLIST[0] if WATCHLIST else None
if TEST_SYMBOL is None:
    raise RuntimeError("No symbols passed the canonical fundamental universe gate.")

au_trades = backtest_one_symbol_nonoverlap(TEST_SYMBOL)

print("\n====================================")
print(f"📊 {TEST_SYMBOL} BACKTEST")
print("====================================")

print(
    f"Signals available : "
    f"{len(historical_signals[historical_signals['Symbol'] == TEST_SYMBOL]):,}"
)

print(f"Trades executed   : {len(au_trades):,}")

if not au_trades.empty:

    print(
        f"Total P&L         : "
        f"{au_trades['PnL'].sum():,.2f} THB"
    )

    print(
        f"Average Return    : "
        f"{au_trades['ReturnPct'].mean():.2f}%"
    )

    print(
        f"Median Return     : "
        f"{au_trades['ReturnPct'].median():.2f}%"
    )

    print(
        f"Win Rate          : "
        f"{(au_trades['ReturnPct'] > 0).mean() * 100:.2f}%"
    )

    print("\n====================================")
    print("🔎 TRADE SAMPLE")
    print("====================================")

    display(
        au_trades[
            [
                "Symbol",
                "SignalDate",
                "EntryDate",
                "EntryPrice",
                "StopPrice",
                "ExitDate",
                "ExitPrice",
                "ExitReason",
                "HoldingDays",
                "Shares",
                "PnL",
                "ReturnPct",
                "Target1Hit",
            ]
        ].head(15)
    )

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    invalid_exit = (
        au_trades["ExitDate"] < au_trades["EntryDate"]
    ).sum()

    invalid_return = (
        ~pd.to_numeric(
            au_trades["ReturnPct"],
            errors="coerce"
        ).notna()
    ).sum()

    invalid_holding = (
        au_trades["HoldingDays"] > MAX_HOLDING_DAYS
    ).sum()

    print("\n====================================")
    print("🛡️ BACKTEST VALIDATION")
    print("====================================")
    print(f"Exit before Entry : {invalid_exit}")
    print(f"Invalid Return    : {invalid_return}")
    print(f"Over Max Holding  : {invalid_holding}")

    assert invalid_exit == 0
    assert invalid_return == 0
    assert invalid_holding == 0

    print("\n====================================")
    print("✅ CELL 12.2 PASSED")
    print("====================================")
    print(
        "Non-overlapping backtest is valid for the test symbol."
    )

else:
    print(
        "\n⚠️ No executable trades found for",
        TEST_SYMBOL
    )



In [ ]:

# ============================================================
# CELL 12.3 — TRADE ENGINE DIAGNOSTICS
# ============================================================

print("=" * 90)
print("🔎 TRADE ENGINE DIAGNOSTICS — TEST SYMBOL")
print("=" * 90)

if au_trades.empty:
    print("⚠️ No trades available.")
else:

    df = au_trades.copy()

    # --------------------------------------------------------
    # 1) Exit distribution
    # --------------------------------------------------------

    exit_counts = (
        df["ExitReason"]
        .value_counts()
        .rename_axis("ExitReason")
        .reset_index(name="Trades")
    )

    exit_counts["Pct"] = (
        exit_counts["Trades"] / len(df) * 100
    )

    print("\n====================================")
    print("📊 EXIT DISTRIBUTION")
    print("====================================")

    display(exit_counts)

    # --------------------------------------------------------
    # 2) Return distribution
    # --------------------------------------------------------

    print("\n====================================")
    print("📈 RETURN DISTRIBUTION")
    print("====================================")

    print(f"Best trade        : {df['ReturnPct'].max():.2f}%")
    print(f"Worst trade       : {df['ReturnPct'].min():.2f}%")
    print(f"Average           : {df['ReturnPct'].mean():.2f}%")
    print(f"Median            : {df['ReturnPct'].median():.2f}%")
    print(f"Win rate          : {(df['ReturnPct'] > 0).mean() * 100:.2f}%")
    print(f"Loss rate         : {(df['ReturnPct'] < 0).mean() * 100:.2f}%")
    print(f"Flat              : {(df['ReturnPct'] == 0).mean() * 100:.2f}%")

    # --------------------------------------------------------
    # 3) Risk characteristics
    # --------------------------------------------------------

    df["RiskPct"] = (
        (df["EntryPrice"] - df["StopPrice"])
        / df["EntryPrice"]
        * 100
    )

    print("\n====================================")
    print("🛡️ STOP / RISK DISTRIBUTION")
    print("====================================")

    print(f"Average Stop Risk : {df['RiskPct'].mean():.2f}%")
    print(f"Median Stop Risk  : {df['RiskPct'].median():.2f}%")
    print(f"Minimum Stop Risk : {df['RiskPct'].min():.2f}%")
    print(f"Maximum Stop Risk : {df['RiskPct'].max():.2f}%")

    # --------------------------------------------------------
    # 4) Target1 reached but eventually lost
    # --------------------------------------------------------

    target1_then_loss = df[
        (df["Target1Hit"] == True) &
        (df["ReturnPct"] < 0)
    ].copy()

    print("\n====================================")
    print("⚠️ TARGET1 HIT → EVENTUAL LOSS")
    print("====================================")

    print(
        f"Count: {len(target1_then_loss):,} "
        f"({len(target1_then_loss) / len(df) * 100:.2f}%)"
    )

    if not target1_then_loss.empty:
        display(
            target1_then_loss[
                [
                    "SignalDate",
                    "EntryDate",
                    "EntryPrice",
                    "StopPrice",
                    "ExitDate",
                    "ExitPrice",
                    "ExitReason",
                    "ReturnPct",
                    "PnL",
                ]
            ].sort_values("ReturnPct").head(10)
        )

    # --------------------------------------------------------
    # 5) Large stop-distance trades
    # --------------------------------------------------------

    print("\n====================================")
    print("⚠️ LARGEST STOP DISTANCES")
    print("====================================")

    display(
        df[
            [
                "SignalDate",
                "EntryDate",
                "EntryPrice",
                "StopPrice",
                "RiskPct",
                "ExitReason",
                "ReturnPct",
                "PnL",
            ]
        ]
        .sort_values("RiskPct", ascending=False)
        .head(10)
    )

    # --------------------------------------------------------
    # 6) Trade duration
    # --------------------------------------------------------

    df["HoldingDays"] = (
        pd.to_datetime(df["ExitDate"])
        - pd.to_datetime(df["EntryDate"])
    ).dt.days

    print("\n====================================")
    print("⏱️ HOLDING PERIOD")
    print("====================================")

    print(f"Average days     : {df['HoldingDays'].mean():.1f}")
    print(f"Median days      : {df['HoldingDays'].median():.1f}")
    print(f"Maximum days     : {df['HoldingDays'].max():.0f}")
    print(f"MAX_HOLDING_DAYS : {MAX_HOLDING_DAYS}")

    over_limit = int(
        (df["HoldingDays"] > MAX_HOLDING_DAYS).sum()
    )

    print(f"Over max holding : {over_limit}")

    # --------------------------------------------------------
    # 7) Basic assertions
    # --------------------------------------------------------

    assert (df["EntryDate"] <= df["ExitDate"]).all()

    assert (df["HoldingDays"] >= 0).all()
    assert (df["HoldingDays"] <= MAX_HOLDING_DAYS).all()
    assert over_limit == 0
    assert df["RiskPct"].notna().all()
    assert df["ReturnPct"].notna().all()

    print("\n====================================")
    print("✅ CELL 12.3 PASSED")
    print("====================================")
    print("Trade-engine diagnostics completed.")


In [ ]:
# ============================================================
# CELL 12.4 — ALL-SYMBOL NON-OVERLAPPING BACKTEST
# Research validation across the complete watchlist
# ============================================================

print("=" * 90)
print("📊 ALL-SYMBOL NON-OVERLAPPING BACKTEST")
print("=" * 90)

all_trades = []
symbol_summary = []

for symbol in WATCHLIST:
    trades = backtest_one_symbol_nonoverlap(symbol)

    signal_count = int(
        (historical_signals["Symbol"] == symbol).sum()
    )

    trade_count = len(trades)

    if not trades.empty:
        all_trades.append(trades)

        symbol_summary.append({
            "Symbol": symbol,
            "Signals": signal_count,
            "Trades": trade_count,
            "PnL": float(trades["PnL"].sum()),
            "AvgReturnPct": float(trades["ReturnPct"].mean()),
            "WinRatePct": float((trades["ReturnPct"] > 0).mean() * 100),
            "MaxHoldingDays": int(trades["HoldingDays"].max()),
        })
    else:
        symbol_summary.append({
            "Symbol": symbol,
            "Signals": signal_count,
            "Trades": 0,
            "PnL": 0.0,
            "AvgReturnPct": np.nan,
            "WinRatePct": np.nan,
            "MaxHoldingDays": 0,
        })

if all_trades:
    all_trades_df = (
        pd.concat(all_trades, ignore_index=True)
        .sort_values(["EntryDate", "Symbol"])
        .reset_index(drop=True)
    )
else:
    all_trades_df = pd.DataFrame()

symbol_summary_df = pd.DataFrame(symbol_summary)

print("\n====================================")
print("📈 PORTFOLIO-LEVEL SUMMARY")
print("====================================")

print(f"Symbols tested     : {len(WATCHLIST)}")
print(f"Total trades       : {len(all_trades_df):,}")

if not all_trades_df.empty:
    print(f"Total P&L          : {all_trades_df['PnL'].sum():,.2f} THB")
    print(f"Average return     : {all_trades_df['ReturnPct'].mean():.2f}%")
    print(f"Median return      : {all_trades_df['ReturnPct'].median():.2f}%")
    print(f"Win rate           : {(all_trades_df['ReturnPct'] > 0).mean() * 100:.2f}%")
    print(f"Max holding days   : {all_trades_df['HoldingDays'].max():.0f}")

    over_limit = int(
        (all_trades_df["HoldingDays"] > MAX_HOLDING_DAYS).sum()
    )

    invalid_dates = int(
        (all_trades_df["ExitDate"] < all_trades_df["EntryDate"]).sum()
    )

    invalid_returns = int(
        (~pd.to_numeric(
            all_trades_df["ReturnPct"],
            errors="coerce"
        ).notna()).sum()
    )

    print(f"Over max holding   : {over_limit}")
    print(f"Invalid exit dates : {invalid_dates}")
    print(f"Invalid returns    : {invalid_returns}")

    print("\n====================================")
    print("🚪 EXIT REASONS")
    print("====================================")
    display(
        all_trades_df["ExitReason"]
        .value_counts()
        .rename_axis("ExitReason")
        .reset_index(name="Trades")
    )

    print("\n====================================")
    print("🏆 BY SYMBOL")
    print("====================================")
    display(
        symbol_summary_df
        .sort_values(["PnL", "WinRatePct"], ascending=[False, False])
        .reset_index(drop=True)
    )

    assert over_limit == 0
    assert invalid_dates == 0
    assert invalid_returns == 0
    assert all_trades_df["HoldingDays"].ge(0).all()
    assert all_trades_df["HoldingDays"].le(MAX_HOLDING_DAYS).all()

    print("\n====================================")
    print("✅ CELL 12.4 PASSED")
    print("====================================")
    print("All-symbol non-overlapping backtest passed validation.")
else:
    print("⚠️ No executable trades found across the watchlist.")


## PRODUCTION PATH — RESEARCH ARCHIVED

Completed research Cells 12.5–12.31 are archived in `research/archive_12.5_to_12.31.ipynb`. They are historical only and are **not part of the Production execution path**.

Do not continue those research branches or promote their variants without a new, explicitly approved out-of-sample research cycle.


In [ ]:
# ============================================================
# CELL 12.32 — PRODUCTION READINESS AUDIT
# ============================================================
print("=" * 90)
print("PRODUCTION READINESS AUDIT — NO STRATEGY CHANGE")
print("=" * 90)

assert "canonical_data" in globals()
assert "PRODUCTION_SYMBOLS" in globals()
assert "PRODUCTION_MIN_SCORE" in globals()
assert "PRODUCTION_WEEKLY_CONFIRM" in globals()
assert "PRODUCTION_ATR_MULT" in globals()
assert "PRODUCTION_TARGET2_PCT" in globals()
assert "PRODUCTION_MAX_HOLDING_DAYS" in globals()

audit_rows = []
for sym in PRODUCTION_SYMBOLS:
    df = canonical_data.get(sym)
    if df is None or df.empty:
        audit_rows.append({
            "Symbol": sym, "Bars": 0, "FirstDate": None, "LastDate": None,
            "MissingOHLCV": True, "EMA75Ready": False, "ATR14Ready": False,
            "Status": "MISSING_DATA",
        })
        continue

    required = ["Open", "High", "Low", "Close", "Volume"]
    missing = bool(df[required].isna().any().any()) if all(c in df.columns for c in required) else True
    bars = len(df)
    ema75_ready = "EMA75" in df.columns and df["EMA75"].notna().sum() >= 1
    atr14_ready = "ATR14" in df.columns and df["ATR14"].notna().sum() >= 1

    audit_rows.append({
        "Symbol": sym,
        "Bars": bars,
        "FirstDate": pd.Timestamp(df.index.min()).date(),
        "LastDate": pd.Timestamp(df.index.max()).date(),
        "MissingOHLCV": missing,
        "EMA75Ready": ema75_ready,
        "ATR14Ready": atr14_ready,
        "Status": "OK" if bars >= 200 and not missing and ema75_ready and atr14_ready else "CHECK",
    })

production_data_audit = pd.DataFrame(audit_rows)
display(production_data_audit)

print("\nPRODUCTION CONFIG")
print("Symbols:", ", ".join(PRODUCTION_SYMBOLS))
print(f"Min score: {PRODUCTION_MIN_SCORE}")
print(f"Weekly confirmation: {PRODUCTION_WEEKLY_CONFIRM}")
print(f"ATR multiplier: {PRODUCTION_ATR_MULT}")
print(f"Target2: {PRODUCTION_TARGET2_PCT:.2%}")
print(f"Max holding days: {PRODUCTION_MAX_HOLDING_DAYS}")

bad = production_data_audit[production_data_audit["Status"] != "OK"]

print("\nDATA READINESS")
print(f"Production symbols checked: {len(production_data_audit)}")
print(f"Ready: {len(production_data_audit) - len(bad)}")
print(f"Needs attention: {len(bad)}")

if bad.empty:
    print("PRODUCTION READINESS: PASS")
else:
    print("PRODUCTION READINESS: CHECK_REQUIRED")

print("PRODUCTION CONFIGURATION UNCHANGED")
print("PRODUCTION READINESS AUDIT COMPLETE")

